In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import  LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
from sklearn.metrics import log_loss, accuracy_score
from sklearn.model_selection import train_test_split

# 1. Downloading and Reading Data

In [ ]:
# Download latest version
path = kagglehub.dataset_download("parisrohan/credit-score-classification")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'credit-score-classification' dataset.
Path to dataset files: /kaggle/input/credit-score-classification


In [ ]:
# Read the dataset
df_train=pd.read_csv(path+"/train.csv")
df_test=pd.read_csv(path+"/test.csv")

/tmp/ipython-input-1863666964.py:2: DtypeWarning: Columns (26) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train=pd.read_csv(path+"/train.csv")


In [ ]:
df_train.head(5)

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,_,809.98,26.822620,22 Years and 1 Months,No,49.574949,80.41529543900253,High_spent_Small_value_payments,312.49408867943663,Good
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.944960,NaN,No,49.574949,118.28022162236736,Low_spent_Large_value_payments,284.62916249607184,Good
2,0x1604,CUS_0xd40,March,Aaron Maashoh,-500,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,28.609352,22 Years and 3 Months,No,49.574949,81.699521264648,Low_spent_Medium_value_payments,331.2098628537912,Good
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,NaN,3,...,Good,809.98,31.377862,22 Years and 4 Months,No,49.574949,199.4580743910713,Low_spent_Small_value_payments,223.45130972736786,Good
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.12,1824.843333,3,...,Good,809.98,24.797347,22 Years and 5 Months,No,49.574949,41.420153086217326,High_spent_Medium_value_payments,341.48923103222177,Good


# 2. Data Exploration


In [ ]:
print(df_train.info())
print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 28 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  object 
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  object 
 8   Monthly_Inhand_Salary     84998 non-null   float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  object 
 13  Type_of_Loan              88592 non-null   ob

In [ ]:
df_train.describe()

,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_Credit_Inquiries,Credit_Utilization_Ratio,Total_EMI_per_month
count,84998.000000,100000.000000,100000.00000,100000.000000,100000.000000,98035.000000,100000.000000,100000.000000
mean,4194.170850,17.091280,22.47443,72.466040,21.068780,27.754251,32.285173,1403.118217
std,3183.686167,117.404834,129.05741,466.422621,14.860104,193.177339,5.116875,8306.041270
min,303.645417,-1.000000,0.00000,1.000000,-5.000000,0.000000,20.000000,0.000000
25%,1625.568229,3.000000,4.00000,8.000000,10.000000,3.000000,28.052567,30.306660
50%,3093.745000,6.000000,5.00000,13.000000,18.000000,6.000000,32.305784,69.249473
75%,5957.448333,7.000000,7.00000,20.000000,28.000000,9.000000,36.496663,161.224249
max,15204.633333,1798.000000,1499.00000,5797.000000,67.000000,2597.000000,50.000000,82331.000000


In [ ]:
df_test.describe()

,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Delay_from_due_date,Num_Credit_Inquiries,Credit_Utilization_Ratio,Total_EMI_per_month
count,42502.000000,50000.000000,50000.000000,50000.000000,50000.000000,48965.000000,50000.000000,50000.000000
mean,4182.004291,16.838260,22.921480,68.772640,21.052640,30.080200,32.279581,1491.304305
std,3174.109304,116.396848,129.314804,451.602363,14.860397,196.984121,5.106238,8595.647887
min,303.645417,-1.000000,0.000000,1.000000,-5.000000,0.000000,20.509652,0.000000
25%,1625.188333,3.000000,4.000000,8.000000,10.000000,4.000000,28.061040,32.222388
50%,3086.305000,6.000000,5.000000,13.000000,18.000000,7.000000,32.280390,74.733349
75%,5934.189094,7.000000,7.000000,20.000000,28.000000,10.000000,36.468591,176.157491
max,15204.633333,1798.000000,1499.000000,5799.000000,67.000000,2593.000000,48.540663,82398.000000


In [ ]:
df_train.duplicated().sum()

np.int64(0)

In [ ]:
for col in df_train.columns:
    print(col, df_train[col].nunique())

ID 100000
Customer_ID 12500
Month 8
Name 10139
Age 1788
SSN 12501
Occupation 16
Annual_Income 18940
Monthly_Inhand_Salary 13235
Num_Bank_Accounts 943
Num_Credit_Card 1179
Interest_Rate 1750
Num_of_Loan 434
Type_of_Loan 6260
Delay_from_due_date 73
Num_of_Delayed_Payment 749
Changed_Credit_Limit 4384
Num_Credit_Inquiries 1223
Credit_Mix 4
Outstanding_Debt 13178
Credit_Utilization_Ratio 100000
Credit_History_Age 404
Payment_of_Min_Amount 3
Total_EMI_per_month 14950
Amount_invested_monthly 91049
Payment_Behaviour 7
Monthly_Balance 98792
Credit_Score 3


In [ ]:
# Calculate the amount of different
for col in df_test.columns:
  if col not in ['ID','Customer_ID','Name','SSN']:
    print(col,df_train[col].dtype, df_train[col].value_counts())
    print('-'*100)

Month object Month
January     12500
February    12500
March       12500
April       12500
May         12500
June        12500
July        12500
August      12500
Name: count, dtype: int64
----------------------------------------------------------------------------------------------------
Age object Age
38      2833
28      2829
31      2806
26      2792
32      2749
        ... 
6618       1
3155       1
5063       1
2875       1
4347       1
Name: count, Length: 1788, dtype: int64
----------------------------------------------------------------------------------------------------
Occupation object Occupation
_______          7062
Lawyer           6575
Architect        6355
Engineer         6350
Scientist        6299
Mechanic         6291
Accountant       6271
Developer        6235
Media_Manager    6232
Teacher          6215
Entrepreneur     6174
Doctor           6087
Journalist       6085
Manager          5973
Musician         5911
Writer           5885
Name: count, dtype: int64
----

#3. Data Cleaning

After initial check, we found that there are many dirty value among multiple columns. So we decide to clean each column which we might use independently.

In [ ]:
# Print each feature’s unique values along with their frequencies.
def print_vlaue_count(col):
  print(df_train[col].value_counts())
  print(df_test[col].value_counts())
  print(df_train[col].dtype)

# Remove underscores from the string-type feature and convert its type to integer.
def replace_astype(col):
  df_train[col]=df_train[col].replace(r'\_', '',regex=True)
  df_test[col]=df_test[col].replace(r'\_', '',regex=True)
  df_train[col]=df_train[col].astype(float).astype(int)
  df_test[col]=df_test[col].astype(float).astype(int)

def capping_quantile(col):
  Q1 = df_train[col].quantile(0.25)
  Q3 = df_train[col].quantile(0.75)
  IQR = Q3 - Q1

  lower = Q1 - 1.5 * IQR
  upper = Q3 + 1.5 * IQR

  df_train[col] = df_train[col].clip(lower, upper)
  df_test[col] = df_test[col].clip(lower, upper)
  print_vlaue_count(col)
def capping_fixed(col,lower,upper):
  df_train[col] = df_train[col].clip(lower, upper)
  df_test[col] = df_test[col].clip(lower, upper)
  print_vlaue_count(col)

**Age**

In [ ]:
# Clean Data with invalid format
# df_train['Age']=df_train['Age'].replace(r'\_', '', regex=True)
# df_train['Age'] = df_train['Age'].astype(float)
# df_test['Age']=df_test['Age'].replace(r'\_', '', regex=True)
# df_test['Age'] = df_test['Age'].astype(float)
# print(df_train['Age'].value_counts())
# print(df_test['Age'].value_counts())
print_vlaue_count('Age')
replace_astype('Age')
capping_fixed('Age',0,100)

Age
38      2833
28      2829
31      2806
26      2792
32      2749
        ... 
6618       1
3155       1
5063       1
2875       1
4347       1
Name: count, Length: 1788, dtype: int64
Age
39      1493
32      1440
44      1428
22      1422
35      1414
        ... 
2698       1
410        1
4955       1
4399       1
5994       1
Name: count, Length: 976, dtype: int64
object
Age
38     2994
28     2968
31     2955
26     2945
32     2884
36     2868
35     2866
25     2861
27     2859
39     2846
34     2837
44     2824
19     2793
41     2785
22     2785
20     2744
37     2742
29     2735
43     2734
30     2727
21     2716
24     2714
23     2654
45     2642
40     2609
42     2577
33     2543
18     2385
100    1891
46     1621
15     1574
17     1502
16     1455
48     1385
49     1375
55     1366
52     1356
53     1354
54     1311
51     1291
50     1273
47     1227
14     1175
0       886
56      362
95        3
99        1
Name: count, dtype: int64
Age
39     1570
32     152

In [ ]:
# Dealing with Outlier data
# df_train['Age']=df_train['Age'].clip(lower=0, upper=100)
# df_test['Age']=df_test['Age'].clip(lower=0, upper=100)
# print(df_train['Age'].value_counts())
# print(df_test['Age'].value_counts())

**Occupation**

In [ ]:
print_vlaue_count('Occupation')

Occupation
_______          7062
Lawyer           6575
Architect        6355
Engineer         6350
Scientist        6299
Mechanic         6291
Accountant       6271
Developer        6235
Media_Manager    6232
Teacher          6215
Entrepreneur     6174
Doctor           6087
Journalist       6085
Manager          5973
Musician         5911
Writer           5885
Name: count, dtype: int64
Occupation
_______          3438
Lawyer           3324
Engineer         3212
Architect        3195
Mechanic         3168
Developer        3146
Accountant       3133
Media_Manager    3130
Scientist        3104
Teacher          3103
Entrepreneur     3103
Journalist       3037
Doctor           3027
Manager          3000
Musician         2947
Writer           2933
Name: count, dtype: int64
object


In [ ]:
df_train['Occupation']=df_train['Occupation'].replace('_______', 'unknown_occupation')
df_test['Occupation']=df_test['Occupation'].replace('_______', 'unknown_occupation')

In [ ]:
print_vlaue_count('Occupation')

Occupation
unknown_occupation    7062
Lawyer                6575
Architect             6355
Engineer              6350
Scientist             6299
Mechanic              6291
Accountant            6271
Developer             6235
Media_Manager         6232
Teacher               6215
Entrepreneur          6174
Doctor                6087
Journalist            6085
Manager               5973
Musician              5911
Writer                5885
Name: count, dtype: int64
Occupation
unknown_occupation    3438
Lawyer                3324
Engineer              3212
Architect             3195
Mechanic              3168
Developer             3146
Accountant            3133
Media_Manager         3130
Scientist             3104
Teacher               3103
Entrepreneur          3103
Journalist            3037
Doctor                3027
Manager               3000
Musician              2947
Writer                2933
Name: count, dtype: int64
object


**Annual Income**

In [ ]:
print_vlaue_count('Annual_Income')
replace_astype('Annual_Income')
capping_quantile('Annual_Income')

Annual_Income
36585.12     16
20867.67     16
17273.83     16
95596.35     15
33029.66     15
             ..
3917169.0     1
89394.78_     1
36806.84_     1
24363.78_     1
95685.21_     1
Name: count, Length: 18940, dtype: int64
Annual_Income
72524.2       8
95596.35      8
36585.12      8
22434.16      8
9141.63       8
             ..
20601508.0    1
29469.98_     1
18940.82_     1
35793.97_     1
31700.3_      1
Name: count, Length: 16121, dtype: int64
object
Annual_Income
152789.5    2783
14293.0       40
35733.0       32
14773.0       32
29020.0       31
            ... 
10409.0        6
19395.0        6
14447.0        6
45253.0        6
108064.0       5
Name: count, Length: 11268, dtype: int64
Annual_Income
152789.5    1394
14293.0       18
14773.0       16
29020.0       16
35733.0       16
            ... 
37827.0        2
68763.0        2
35579.0        2
37972.0        2
49099.0        1
Name: count, Length: 11268, dtype: int64
float64


In [ ]:
# Clean Outlier data
print_vlaue_count('Annual_Income')

Annual_Income
152789.5    2783
14293.0       40
35733.0       32
14773.0       32
29020.0       31
            ... 
10409.0        6
19395.0        6
14447.0        6
45253.0        6
108064.0       5
Name: count, Length: 11268, dtype: int64
Annual_Income
152789.5    1394
14293.0       18
14773.0       16
29020.0       16
35733.0       16
            ... 
37827.0        2
68763.0        2
35579.0        2
37972.0        2
49099.0        1
Name: count, Length: 11268, dtype: int64
float64


**Monthly_Inhand_Salary**

In [ ]:
print_vlaue_count('Monthly_Inhand_Salary')
capping_quantile('Monthly_Inhand_Salary')

Monthly_Inhand_Salary
2295.058333    15
6082.187500    15
6358.956667    15
6769.130000    15
3080.555000    14
               ..
3415.781667     1
6272.739429     1
1069.950000     1
454.382083      1
2319.831269     1
Name: count, Length: 13235, dtype: int64
Monthly_Inhand_Salary
1315.560833    8
3080.555000    7
536.431250     7
5766.491667    7
2295.058333    7
              ..
699.392713     1
2872.802500    1
1462.177500    1
6148.627500    1
1401.694167    1
Name: count, Length: 12793, dtype: int64
float64
Monthly_Inhand_Salary
12455.268490    1683
6082.187500       15
6769.130000       15
6358.956667       15
2295.058333       15
                ... 
817.769538         1
5712.091000        1
2298.312500        1
1211.935000        1
1234.070825        1
Name: count, Length: 12972, dtype: int64
Monthly_Inhand_Salary
12455.268490    833
1315.560833       8
6639.560000       7
3080.555000       7
4387.272500       7
               ... 
5847.434739       1
2872.802500       1
1462.

In [ ]:
df_train['Monthly_Inhand_Salary'].isnull().mean()

np.float64(0.15002)

In [ ]:
df_train['Monthly_Inhand_Salary']=df_train['Monthly_Inhand_Salary'].fillna(df_train['Monthly_Inhand_Salary'].mean())
df_test['Monthly_Inhand_Salary']=df_test['Monthly_Inhand_Salary'].fillna(df_test['Monthly_Inhand_Salary'].mean())

**Num_Bank_Accounts**

In [ ]:
print_vlaue_count('Num_Bank_Accounts')
capping_quantile('Num_Bank_Accounts')


Num_Bank_Accounts
6       13001
7       12823
8       12765
4       12186
5       12118
        ...  
1091        1
1123        1
1657        1
299         1
1240        1
Name: count, Length: 943, dtype: int64
Num_Bank_Accounts
6       6504
7       6408
8       6387
4       6100
5       6068
        ... 
800        1
976        1
207        1
1395       1
802        1
Name: count, Length: 540, dtype: int64
int64
Num_Bank_Accounts
 6     13001
 7     12823
 8     12765
 4     12186
 5     12118
 3     11950
 9      5443
 10     5247
 1      4490
 0      4328
 2      4304
 13     1315
-1        21
 11        9
Name: count, dtype: int64
Num_Bank_Accounts
 6     6504
 7     6408
 8     6387
 4     6100
 5     6068
 3     5955
 9     2738
 10    2599
 1     2253
 0     2166
 2     2152
 13     635
 11      19
-1       16
Name: count, dtype: int64
int64


In [ ]:
capping_fixed('Num_Bank_Accounts',0,20)

Num_Bank_Accounts
6     13001
7     12823
8     12765
4     12186
5     12118
3     11950
9      5443
10     5247
1      4490
0      4349
2      4304
13     1315
11        9
Name: count, dtype: int64
Num_Bank_Accounts
6     6504
7     6408
8     6387
4     6100
5     6068
3     5955
9     2738
10    2599
1     2253
0     2182
2     2152
13     635
11      19
Name: count, dtype: int64
int64


**Num_Credit_Card**

In [ ]:
print_vlaue_count('Num_Credit_Card')

Num_Credit_Card
5       18459
7       16615
6       16559
4       14030
3       13277
        ...  
1405        1
708         1
62          1
343         1
481         1
Name: count, Length: 1179, dtype: int64
Num_Credit_Card
5       9210
7       8271
6       8243
4       7072
3       6539
        ... 
1209       1
934        1
417        1
94         1
601        1
Name: count, Length: 819, dtype: int64
int64


In [ ]:
capping_quantile('Num_Credit_Card')

Num_Credit_Card
5.0     18459
7.0     16615
6.0     16559
4.0     14030
3.0     13277
8.0      4956
10.0     4860
9.0      4643
11.5     2271
2.0      2149
1.0      2132
11.0       36
0.0        13
Name: count, dtype: int64
Num_Credit_Card
5.0     9210
7.0     8271
6.0     8243
4.0     7072
3.0     6539
8.0     2497
10.0    2405
9.0     2333
11.5    1179
2.0     1131
1.0     1063
11.0      41
0.0       16
Name: count, dtype: int64
float64


In [ ]:
df_train['Num_Credit_Card']=df_train['Num_Credit_Card'].astype(int)
df_test['Num_Credit_Card']=df_test['Num_Credit_Card'].astype(int)

**Interest_Rate**

In [ ]:
print_vlaue_count('Interest_Rate')

Interest_Rate
8       5012
5       4979
6       4721
10      4540
12      4540
        ... 
2548       1
967        1
3790       1
3782       1
4372       1
Name: count, Length: 1750, dtype: int64
Interest_Rate
8       2503
5       2500
6       2368
12      2288
10      2259
        ... 
3279       1
1166       1
5613       1
2304       1
3267       1
Name: count, Length: 945, dtype: int64
int64


In [ ]:
capping_quantile('Interest_Rate')

Interest_Rate
8     5012
5     4979
6     4721
10    4540
12    4540
9     4494
7     4494
11    4428
18    4102
15    3992
20    3929
17    3813
16    3730
19    3630
3     2765
1     2683
4     2589
2     2465
13    2384
14    2229
38    2034
32    1742
22    1720
30    1690
24    1685
23    1683
29    1662
28    1616
27    1608
25    1566
21    1560
34    1502
26    1489
33    1467
31    1457
Name: count, dtype: int64
Interest_Rate
8     2503
5     2500
6     2368
12    2288
10    2259
9     2253
7     2250
11    2198
18    2052
15    1992
20    1961
17    1906
16    1867
19    1810
3     1388
1     1344
4     1287
2     1245
13    1187
14    1122
38     966
32     874
22     860
24     848
23     847
30     846
29     833
28     815
27     808
25     790
21     775
26     749
34     744
33     734
31     731
Name: count, dtype: int64
int64


**Num_of_Loan**

In [ ]:
print_vlaue_count('Num_of_Loan')

Num_of_Loan
3      14386
2      14250
4      14016
0      10380
1      10083
       ...  
41         1
18         1
56         1
657        1
917        1
Name: count, Length: 434, dtype: int64
Num_of_Loan
2       7173
3       7114
4       6982
0       5163
1       5029
        ... 
1304       1
569        1
350        1
1221       1
799        1
Name: count, Length: 263, dtype: int64
object


In [ ]:
replace_astype('Num_of_Loan')

In [ ]:
capping_quantile('Num_of_Loan')

Num_of_Loan
 3     15104
 2     15032
 4     14743
 0     10930
 1     10606
 6      7803
 7      7344
 5      7197
-5      3876
 9      3702
 8      3191
 11      472
Name: count, dtype: int64
Num_of_Loan
 2     7515
 3     7514
 4     7368
 0     5446
 1     5295
 6     3902
 7     3680
 5     3617
-5     1974
 9     1837
 8     1594
 11     258
Name: count, dtype: int64
int64


In [ ]:
capping_fixed('Num_of_Loan',0,11)

Num_of_Loan
3     15104
2     15032
0     14806
4     14743
1     10606
6      7803
7      7344
5      7197
9      3702
8      3191
11      472
Name: count, dtype: int64
Num_of_Loan
2     7515
3     7514
0     7420
4     7368
1     5295
6     3902
7     3680
5     3617
9     1837
8     1594
11     258
Name: count, dtype: int64
int64


**Type of Loan**

In [ ]:
print_vlaue_count('Type_of_Loan')
df_train['Type_of_Loan'].value_counts().head(20)

Type_of_Loan
Not Specified                                                                                                                    1408
Credit-Builder Loan                                                                                                              1280
Personal Loan                                                                                                                    1272
Debt Consolidation Loan                                                                                                          1264
Student Loan                                                                                                                     1240
                                                                                                                                 ... 
Debt Consolidation Loan, Personal Loan, Mortgage Loan, Personal Loan, Not Specified, Mortgage Loan, and Home Equity Loan            8
Student Loan, Home Equity Loan, Student Loan, Per

,count
Type_of_Loan,
Not Specified,1408
Credit-Builder Loan,1280
Personal Loan,1272
Debt Consolidation Loan,1264
Student Loan,1240
Payday Loan,1200
Mortgage Loan,1176
Auto Loan,1152
Home Equity Loan,1136


In [ ]:
df_train['Credit-Builder Loan']=df_train['Type_of_Loan'].str.contains('Credit-Builder Loan').fillna(False)
df_train['Personal Loan']=df_train['Type_of_Loan'].str.contains('Personal Loan').fillna(False)
df_train['Student Loan']=df_train['Type_of_Loan'].str.contains('Student Loan').fillna(False)
df_train['Mortgage']=df_train['Type_of_Loan'].str.contains('Mortgage').fillna(False)
df_train['Home Equity Loan']=df_train['Type_of_Loan'].str.contains('Home Equity Loan').fillna(False)
df_train['Payday Loan']=df_train['Type_of_Loan'].str.contains('Payday Loan').fillna(False)
df_train['Other Loan']=df_train['Type_of_Loan'].str.contains('Other Loan').fillna(False)
df_train['Debt Consolidation']=df_train['Type_of_Loan'].str.contains('Debt Consolidation Loan').fillna(False)

df_test['Credit-Builder Loan']=df_test['Type_of_Loan'].str.contains('Credit-Builder Loan').fillna(False)
df_test['Personal Loan']=df_test['Type_of_Loan'].str.contains('Personal Loan').fillna(False)
df_test['Student Loan']=df_test['Type_of_Loan'].str.contains('Student Loan').fillna(False)
df_test['Mortgage']=df_test['Type_of_Loan'].str.contains('Mortgage').fillna(False)
df_test['Home Equity Loan']=df_test['Type_of_Loan'].str.contains('Home Equity Loan').fillna(False)
df_test['Payday Loan']=df_test['Type_of_Loan'].str.contains('Payday Loan').fillna(False)
df_test['Other Loan']=df_test['Type_of_Loan'].str.contains('Other Loan').fillna(False)
df_test['Debt Consolidation']=df_test['Type_of_Loan'].str.contains('Debt Consolidation Loan').fillna(False)



/tmp/ipython-input-99666179.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_train['Credit-Builder Loan']=df_train['Type_of_Loan'].str.contains('Credit-Builder Loan').fillna(False)
/tmp/ipython-input-99666179.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_train['Personal Loan']=df_train['Type_of_Loan'].str.contains('Personal Loan').fillna(False)
/tmp/ipython-input-99666179.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) i

**Delay_from_due_date**

In [ ]:
print_vlaue_count('Delay_from_due_date')

Delay_from_due_date
 15    3596
 13    3424
 8     3324
 14    3313
 10    3281
       ... 
-4       62
 65      56
-5       33
 66      32
 67      22
Name: count, Length: 73, dtype: int64
Delay_from_due_date
 13    1761
 15    1759
 8     1680
 9     1656
 10    1645
       ... 
 65      30
 63      21
-5       18
 66      12
 67       7
Name: count, Length: 73, dtype: int64
int64


In [ ]:
capping_quantile('Delay_from_due_date')
capping_fixed('Delay_from_due_date',0,100)

Delay_from_due_date
 55    4562
 15    3596
 13    3424
 8     3324
 14    3313
       ... 
-1      210
-2      168
-3      118
-4       62
-5       33
Name: count, Length: 61, dtype: int64
Delay_from_due_date
 55    2292
 13    1761
 15    1759
 8     1680
 9     1656
       ... 
-1      101
-2       71
-3       59
-4       49
-5       18
Name: count, Length: 61, dtype: int64
int64
Delay_from_due_date
55    4562
15    3596
13    3424
8     3324
14    3313
10    3281
7     3234
9     3233
11    3182
12    3141
6     3137
5     3042
19    2638
18    2637
27    2623
16    2566
24    2533
17    2524
25    2506
20    2489
21    2411
28    2397
23    2387
26    2386
29    2383
22    2334
30    2309
0     1786
4     1722
3     1686
2     1342
1     1326
31     802
33     791
32     787
34     675
47     654
48     628
52     625
54     624
42     623
35     614
44     602
36     594
38     592
41     586
53     585
50     576
40     572
49     538
45     536
51     535
39     525
43     502


**Num_of_Delayed_Payment**

In [ ]:
print_vlaue_count('Num_of_Delayed_Payment')

Num_of_Delayed_Payment
19      5327
17      5261
16      5173
10      5153
18      5083
        ... 
3845       1
4075       1
1502       1
1530       1
3011       1
Name: count, Length: 749, dtype: int64
Num_of_Delayed_Payment
19      2622
15      2594
18      2570
16      2548
17      2545
        ... 
3178       1
1122       1
959        1
272        1
3594       1
Name: count, Length: 443, dtype: int64
object


In [ ]:
df_train['Num_of_Delayed_Payment']=df_train['Num_of_Delayed_Payment'].fillna(0)
df_test['Num_of_Delayed_Payment']=df_test['Num_of_Delayed_Payment'].fillna(0)

In [ ]:
replace_astype('Num_of_Delayed_Payment')

In [ ]:
capping_quantile('Num_of_Delayed_Payment')
capping_fixed('Num_of_Delayed_Payment',0,100)

Num_of_Delayed_Payment
 0     8611
 19    5481
 17    5412
 16    5312
 10    5309
 15    5237
 18    5216
 20    5089
 12    5059
 9     4981
 8     4873
 11    4810
 14    4193
 13    4036
 21    2553
 7     2385
 22    2339
 6     2321
 5     2091
 23    2028
 3     1931
 4     1838
 2     1810
 24    1701
 25    1665
 1     1636
 33     736
 26     322
-1      316
 27     250
-2      234
 28     131
-3       94
Name: count, dtype: int64
Num_of_Delayed_Payment
 0     4309
 19    2707
 15    2674
 16    2637
 17    2636
 18    2631
 10    2591
 12    2563
 20    2518
 11    2504
 9     2440
 8     2430
 14    2066
 13    2061
 21    1350
 7     1186
 22    1148
 6     1114
 5     1063
 23    1039
 3      967
 4      909
 2      906
 24     856
 1      849
 25     845
 33     395
 26     150
-1      127
-2      109
 27     104
 28      65
-3       51
Name: count, dtype: int64
int64
Num_of_Delayed_Payment
0     9255
19    5481
17    5412
16    5312
10    5309
15    5237
18    5216
20  

**Changed_Credit_Limit**

In [ ]:
print_vlaue_count('Changed_Credit_Limit')

Changed_Credit_Limit
_                    2091
8.22                  133
11.5                  127
11.32                 126
7.35                  121
                     ... 
30.16                   1
4.710000000000001       1
-4.39                   1
27.38                   1
16.63                   1
Name: count, Length: 4384, dtype: int64
Changed_Credit_Limit
_                     1059
11.5                    70
11.32                   63
7.35                    60
7.01                    60
                      ... 
24.3                     1
1.5699999999999998       1
-1.11                    1
27.38                    1
35.53                    1
Name: count, Length: 3927, dtype: int64
object


In [ ]:
df_train['Changed_Credit_Limit']=df_train['Changed_Credit_Limit'].replace(r'[_\-]', '',regex=True)
df_test['Changed_Credit_Limit']=df_test['Changed_Credit_Limit'].replace(r'[_\-]', '',regex=True)


In [ ]:
df_train['Changed_Credit_Limit']=pd.to_numeric(df_train['Changed_Credit_Limit'],errors='coerce')
df_test['Changed_Credit_Limit']=pd.to_numeric(df_test['Changed_Credit_Limit'],errors='coerce')

In [ ]:
df_train['Changed_Credit_Limit']=df_train['Changed_Credit_Limit'].fillna(df_train['Changed_Credit_Limit'].mean())
df_test['Changed_Credit_Limit']=df_test['Changed_Credit_Limit'].fillna(df_test['Changed_Credit_Limit'].mean())

**Num_Credit_Inquiries**

In [ ]:
print_vlaue_count('Num_Credit_Inquiries')

Num_Credit_Inquiries
4.0       11271
3.0        8890
6.0        8111
7.0        8058
2.0        8028
          ...  
1618.0        1
758.0         1
735.0         1
2483.0        1
1960.0        1
Name: count, Length: 1223, dtype: int64
Num_Credit_Inquiries
5.0       4709
4.0       4402
6.0       4375
7.0       4295
8.0       3922
          ... 
649.0        1
26.0         1
2070.0       1
1292.0       1
1634.0       1
Name: count, Length: 750, dtype: int64
float64


In [ ]:
df_train['Num_Credit_Inquiries']=df_train['Num_Credit_Inquiries'].fillna(df_train['Num_Credit_Inquiries'].mean())
df_test['Num_Credit_Inquiries']=df_test['Num_Credit_Inquiries'].fillna(df_test['Num_Credit_Inquiries'].mean())

**Credit Mix**

In [ ]:
print_vlaue_count('Credit_Mix')

Credit_Mix
Standard    36479
Good        24337
_           20195
Bad         18989
Name: count, dtype: int64
Credit_Mix
Standard    18379
Good        12260
_            9805
Bad          9556
Name: count, dtype: int64
object


In [ ]:
df_train['Credit_Mix']=df_train['Credit_Mix'].replace('_','unknown_credit_mix')
df_test['Credit_Mix']=df_test['Credit_Mix'].replace('_','unknown_credit_mix')

**Outstanding_Debt**

In [ ]:
print_vlaue_count('Outstanding_Debt')

Outstanding_Debt
1360.45     24
1151.7      23
460.46      23
1109.03     23
1329.59     16
            ..
1619.56_     1
297.64_      1
1264.42_     1
1617.55_     1
1324.1_      1
Name: count, Length: 13178, dtype: int64
Outstanding_Debt
460.46      12
1109.03     12
1151.7      12
1360.45     12
978.3        8
            ..
1527.67_     1
853.94_      1
753.21_      1
58.82_       1
1453.24_     1
Name: count, Length: 12685, dtype: int64
object


In [ ]:
df_train['Outstanding_Debt']=df_train['Outstanding_Debt'].replace(r'[_\-]', '',regex=True)
df_test['Outstanding_Debt']=df_test['Outstanding_Debt'].replace(r'[_\-]', '',regex=True)


In [ ]:
df_train['Outstanding_Debt']=pd.to_numeric(df_train['Outstanding_Debt'],errors='coerce')
df_test['Outstanding_Debt']=pd.to_numeric(df_test['Outstanding_Debt'],errors='coerce')

**Credit_Utilization_Ratio**

In [ ]:
print_vlaue_count('Credit_Utilization_Ratio')

Credit_Utilization_Ratio
39.300980    1
38.850680    1
37.753013    1
27.495263    1
36.979007    1
            ..
24.797347    1
31.377862    1
28.609352    1
31.944960    1
26.822620    1
Name: count, Length: 100000, dtype: int64
Credit_Utilization_Ratio
33.086814    1
33.151263    1
27.122224    1
29.324052    1
24.115758    1
            ..
25.926822    1
32.430559    1
33.811894    1
33.053114    1
35.030402    1
Name: count, Length: 50000, dtype: int64
float64


In [ ]:
replace_astype('Credit_Utilization_Ratio')

**Credit_History_Age**

In [ ]:
print_vlaue_count('Credit_History_Age')

Credit_History_Age
15 Years and 11 Months    446
19 Years and 4 Months     445
19 Years and 5 Months     444
17 Years and 11 Months    443
19 Years and 3 Months     441
                         ... 
0 Years and 3 Months       20
0 Years and 2 Months       15
33 Years and 7 Months      14
33 Years and 8 Months      12
0 Years and 1 Months        2
Name: count, Length: 404, dtype: int64
Credit_History_Age
20 Years and 1 Months     254
16 Years and 1 Months     254
18 Years and 7 Months     252
19 Years and 7 Months     252
18 Years and 6 Months     250
                         ... 
4 Years and 5 Months       21
0 Years and 11 Months      16
33 Years and 11 Months     15
34 Years and 0 Months      14
0 Years and 10 Months      13
Name: count, Length: 399, dtype: int64
object


In [ ]:
df_test['Credit_History_Age']=df_test['Credit_History_Age'].fillna("0 Years and 0 Months")
df_train['Credit_History_Age']=df_train['Credit_History_Age'].fillna("0 Years and 0 Months")


In [ ]:
df_test['Credit_History_Age'] = (
    df_test['Credit_History_Age'].str.extract(r'(\d+)\s+Years.*?(\d+)\s+Months')
    .astype(int)
    .apply(lambda x: x[0] * 12 + x[1], axis=1)
)
df_train['Credit_History_Age'] = (
    df_train['Credit_History_Age'].str.extract(r'(\d+)\s+Years.*?(\d+)\s+Months')
    .astype(int)
    .apply(lambda x: x[0] * 12 + x[1], axis=1)
)



**Payment_of_Min_Amount**

In [ ]:
print_vlaue_count('Payment_of_Min_Amount')

Payment_of_Min_Amount
Yes    52326
No     35667
NM     12007
Name: count, dtype: int64
Payment_of_Min_Amount
Yes    26158
No     17849
NM      5993
Name: count, dtype: int64
object


In [ ]:
df_train['Payment_of_Min_Amount']=df_train['Payment_of_Min_Amount'].replace('NM','No')
df_test['Payment_of_Min_Amount']=df_test['Payment_of_Min_Amount'].replace('NM','No')

**Total_EMI_per_month**

In [ ]:
print_vlaue_count('Total_EMI_per_month')

Total_EMI_per_month
0.000000        10613
135.133799          8
182.585183          8
427.144183          8
83.829111           8
                ...  
33746.000000        1
39347.000000        1
27292.000000        1
16627.000000        1
49430.000000        1
Name: count, Length: 14950, dtype: int64
Total_EMI_per_month
0.000000        5002
360.341470         4
78.837730          4
224.884050         4
17.288404          4
                ... 
67300.000000       1
311.984822         1
69805.000000       1
32493.000000       1
178.338617         1
Name: count, Length: 13144, dtype: int64
float64


**Amount_invested_monthly**

In [ ]:
print_vlaue_count('Amount_invested_monthly')

Amount_invested_monthly
__10000__             4305
0.0                    169
59.93725850034815        1
165.180659491917         1
62.030802602004044       1
                      ... 
109.296681189146         1
33.6098814431885         1
76.87001005130772        1
908.6939096189257        1
401.35900899207513       1
Name: count, Length: 91049, dtype: int64
Amount_invested_monthly
__10000__             2175
0.0                    106
121.66817445608021       1
170.15073787772732       1
320.4566446914704        1
                      ... 
366.23148415217315       1
34.89940643392877        1
256.90830529853173       1
41.62264908541225        1
189.03476671896445       1
Name: count, Length: 45450, dtype: int64
object


In [ ]:
df_train['Amount_invested_monthly']=df_train['Amount_invested_monthly'].fillna(0)
df_test['Amount_invested_monthly']=df_test['Amount_invested_monthly'].fillna(0)

In [ ]:
replace_astype('Amount_invested_monthly')

**Payment_Behaviour**

In [ ]:
print_vlaue_count('Payment_Behaviour')

Payment_Behaviour
Low_spent_Small_value_payments      25513
High_spent_Medium_value_payments    17540
Low_spent_Medium_value_payments     13861
High_spent_Large_value_payments     13721
High_spent_Small_value_payments     11340
Low_spent_Large_value_payments      10425
!@9#%8                               7600
Name: count, dtype: int64
Payment_Behaviour
Low_spent_Small_value_payments      12694
High_spent_Medium_value_payments     8922
High_spent_Large_value_payments      6844
Low_spent_Medium_value_payments      6837
High_spent_Small_value_payments      5651
Low_spent_Large_value_payments       5252
!@9#%8                               3800
Name: count, dtype: int64
object


In [ ]:
df_train['Payment_Behaviour']=df_train['Payment_Behaviour'].replace('!@9#%8', 'unknown_payment_behaviour')
df_test['Payment_Behaviour']=df_test['Payment_Behaviour'].replace('!@9#%8', 'unknown_payment_behaviour')

**Monthly_Balance**

In [ ]:
print_vlaue_count('Monthly_Balance')

Monthly_Balance
__-333333333333333333333333333__    9
252.08489793906085                  1
254.9709216273975                   1
250.0931678204641                   1
289.7550752754317                   1
                                   ..
278.8720257394474                   1
376.7024623690405                   1
321.2336043357731                   1
373.29270287694055                  1
336.6371802877606                   1
Name: count, Length: 98792, dtype: int64
Monthly_Balance
__-333333333333333333333333333__    6
223.40782977501067                  1
252.53294533915363                  1
305.43786598764547                  1
389.53754307843735                  1
                                   ..
194.44026842190848                  1
299.957837924029                    1
375.897928536186                    1
313.7383005238995                   1
356.57013660245383                  1
Name: count, Length: 49433, dtype: int64
object


In [ ]:
df_train['Monthly_Balance']=df_train['Monthly_Balance'].fillna(0)
df_test['Monthly_Balance']=df_test['Monthly_Balance'].fillna(0)

In [ ]:
replace_astype('Monthly_Balance')

In [ ]:
#Check result
print(df_train.info())
print(df_test.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 36 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   ID                        100000 non-null  object 
 1   Customer_ID               100000 non-null  object 
 2   Month                     100000 non-null  object 
 3   Name                      90015 non-null   object 
 4   Age                       100000 non-null  int64  
 5   SSN                       100000 non-null  object 
 6   Occupation                100000 non-null  object 
 7   Annual_Income             100000 non-null  float64
 8   Monthly_Inhand_Salary     100000 non-null  float64
 9   Num_Bank_Accounts         100000 non-null  int64  
 10  Num_Credit_Card           100000 non-null  int64  
 11  Interest_Rate             100000 non-null  int64  
 12  Num_of_Loan               100000 non-null  int64  
 13  Type_of_Loan              88592 non-null   ob

In [ ]:
df_train

,ID,Customer_ID,Month,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,...,Monthly_Balance,Credit_Score,Credit-Builder Loan,Personal Loan,Student Loan,Mortgage,Home Equity Loan,Payday Loan,Other Loan,Debt Consolidation
0,0x1602,CUS_0xd40,January,Aaron Maashoh,23,821-00-0265,Scientist,19114.0,1824.843333,3,...,312,Good,True,True,False,False,True,False,False,False
1,0x1603,CUS_0xd40,February,Aaron Maashoh,23,821-00-0265,Scientist,19114.0,4167.185807,3,...,284,Good,True,True,False,False,True,False,False,False
2,0x1604,CUS_0xd40,March,Aaron Maashoh,0,821-00-0265,Scientist,19114.0,4167.185807,3,...,331,Good,True,True,False,False,True,False,False,False
3,0x1605,CUS_0xd40,April,Aaron Maashoh,23,821-00-0265,Scientist,19114.0,4167.185807,3,...,223,Good,True,True,False,False,True,False,False,False
4,0x1606,CUS_0xd40,May,Aaron Maashoh,23,821-00-0265,Scientist,19114.0,1824.843333,3,...,341,Good,True,True,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0x25fe9,CUS_0x942c,April,Nicks,25,078-73-5990,Mechanic,39628.0,3359.415833,4,...,479,Poor,False,False,True,False,False,False,False,False
99996,0x25fea,CUS_0x942c,May,Nicks,25,078-73-5990,Mechanic,39628.0,3359.415833,4,...,496,Poor,False,False,True,False,False,False,False,False
99997,0x25feb,CUS_0x942c,June,Nicks,25,078-73-5990,Mechanic,39628.0,3359.415833,4,...,516,Poor,False,False,True,False,False,False,False,False
99998,0x25fec,CUS_0x942c,July,Nicks,25,078-73-5990,Mechanic,39628.0,3359.415833,4,...,319,Standard,False,False,True,False,False,False,False,False


In [ ]:
from google.colab import drive
import pandas as pd
import os

# 挂载您的 Google Drive
print("正在挂载 Google Drive...")
drive.mount('/content/drive')
print("Google Drive 挂载成功！")

# 确认您的 Drive 路径
# 您的 Google Drive 根目录位于 /content/drive/MyDrive
DRIVE_ROOT = '/content/drive/MyDrive'
# 创建一个示例 DataFrame

df = df_train

print("\n要保存的 DataFrame 如下：")
print(df)
# 定义目标文件夹和文件名

FILE_NAME = 'cleaned_train_data.csv'

# 构建完整的保存路径
# 例如: /content/drive/MyDrive/Colab_Output_Data/my_data_export.csv
SAVE_PATH = os.path.join(DRIVE_ROOT,  FILE_NAME)



# 将 DataFrame 保存为 CSV 文件
# index=False 表示不将 DataFrame 的索引（0, 1, 2, 3...）写入 CSV 文件
df.to_csv(SAVE_PATH, index=False, encoding='utf-8')

print(f"\n✅ 文件已成功保存到 Google Drive：")
print(f"路径: {SAVE_PATH}")

正在挂载 Google Drive...
Mounted at /content/drive
Google Drive 挂载成功！

要保存的 DataFrame 如下：
            ID Customer_ID     Month           Name  Age          SSN  \
0       0x1602   CUS_0xd40   January  Aaron Maashoh   23  821-00-0265   
1       0x1603   CUS_0xd40  February  Aaron Maashoh   23  821-00-0265   
2       0x1604   CUS_0xd40     March  Aaron Maashoh    0  821-00-0265   
3       0x1605   CUS_0xd40     April  Aaron Maashoh   23  821-00-0265   
4       0x1606   CUS_0xd40       May  Aaron Maashoh   23  821-00-0265   
...        ...         ...       ...            ...  ...          ...   
99995  0x25fe9  CUS_0x942c     April          Nicks   25  078-73-5990   
99996  0x25fea  CUS_0x942c       May          Nicks   25  078-73-5990   
99997  0x25feb  CUS_0x942c      June          Nicks   25  078-73-5990   
99998  0x25fec  CUS_0x942c      July          Nicks   25  078-73-5990   
99999  0x25fed  CUS_0x942c    August          Nicks   25  078-73-5990   

      Occupation  Annual_Income  Mont

# 4. Feature Engineering

## V0-Feature Engineering( One-hot + Standardization)

###Handling Categorical Features(One-hot)

In [ ]:
from re import X
#For target variables, I also user label encoding
le = LabelEncoder()
y_train_origin = le.fit_transform(df_train['Credit_Score'])
X_train = df_train.drop(['ID','Customer_ID','Name','SSN','Type_of_Loan','Credit_Score'],axis=1)
X_test = df_test.drop(['ID','Customer_ID','Name','SSN','Type_of_Loan'],axis=1)

In [ ]:
df_train['Credit_Score']

,Credit_Score
0,Good
1,Good
2,Good
3,Good
4,Good
...,...
99995,Poor
99996,Poor
99997,Poor
99998,Standard


In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 30 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   Month                     100000 non-null  object 
 1   Age                       100000 non-null  int64  
 2   Occupation                100000 non-null  object 
 3   Annual_Income             100000 non-null  float64
 4   Monthly_Inhand_Salary     100000 non-null  float64
 5   Num_Bank_Accounts         100000 non-null  int64  
 6   Num_Credit_Card           100000 non-null  int64  
 7   Interest_Rate             100000 non-null  int64  
 8   Num_of_Loan               100000 non-null  int64  
 9   Delay_from_due_date       100000 non-null  int64  
 10  Num_of_Delayed_Payment    100000 non-null  int64  
 11  Changed_Credit_Limit      100000 non-null  float64
 12  Num_Credit_Inquiries      100000 non-null  float64
 13  Credit_Mix                100000 non-null  ob

In [ ]:
X_train.head()

,Month,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,...,Payment_Behaviour,Monthly_Balance,Credit-Builder Loan,Personal Loan,Student Loan,Mortgage,Home Equity Loan,Payday Loan,Other Loan,Debt Consolidation
0,January,23,Scientist,19114.0,1824.843333,3,4,3,4,3,...,High_spent_Small_value_payments,312,True,True,False,False,True,False,False,False
1,February,23,Scientist,19114.0,4167.185807,3,4,3,4,0,...,Low_spent_Large_value_payments,284,True,True,False,False,True,False,False,False
2,March,0,Scientist,19114.0,4167.185807,3,4,3,4,3,...,Low_spent_Medium_value_payments,331,True,True,False,False,True,False,False,False
3,April,23,Scientist,19114.0,4167.185807,3,4,3,4,5,...,Low_spent_Small_value_payments,223,True,True,False,False,True,False,False,False
4,May,23,Scientist,19114.0,1824.843333,3,4,3,4,6,...,High_spent_Medium_value_payments,341,True,True,False,False,True,False,False,False


In [ ]:
#For non target variables which are all nominal, I would use onehot encoding.
def onehot_encoding(X_train,X_test):

  cat_cols=X_train.select_dtypes(include=['object','bool']).columns
  print(cat_cols)

  encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
  encoder.fit(X_train[cat_cols])
  # #Split catergorical and numerical features
  # X_train_num = X_train.select_dtypes(include=['float64', 'int64'])
  # X_test_num = X_test.select_dtypes(include=['float64', 'int64'])


  #Encode catergorical features
  X_train_cat_encoded = pd.DataFrame(encoder.transform(X_train[cat_cols]),
                                   columns=encoder.get_feature_names_out(cat_cols),
                                   index=X_train.index)
  X_test_cat_encoded = pd.DataFrame(encoder.transform(X_test[cat_cols]),
                                  columns=encoder.get_feature_names_out(cat_cols),
                                  index=X_test.index)

  #Split encoded catergorical features and numerical features


  print(X_test_cat_encoded)
  print(X_train_cat_encoded)

  return  X_train_cat_encoded, X_test_cat_encoded

In [ ]:
#Apply the function to X_train and X_test
X_train_cat_encoded, X_test_cat_encoded=onehot_encoding(X_train,X_test)

Index(['Month', 'Occupation', 'Credit_Mix', 'Payment_of_Min_Amount',
       'Payment_Behaviour', 'Credit-Builder Loan', 'Personal Loan',
       'Student Loan', 'Mortgage', 'Home Equity Loan', 'Payday Loan',
       'Other Loan', 'Debt Consolidation'],
      dtype='object')
       Month_April  Month_August  Month_February  Month_January  Month_July  \
0              0.0           0.0             0.0            0.0         0.0   
1              0.0           0.0             0.0            0.0         0.0   
2              0.0           0.0             0.0            0.0         0.0   
3              0.0           0.0             0.0            0.0         0.0   
4              0.0           0.0             0.0            0.0         0.0   
...            ...           ...             ...            ...         ...   
49995          0.0           0.0             0.0            0.0         0.0   
49996          0.0           0.0             0.0            0.0         0.0   
49997          0

### Handling Numerical Features( Standadizaiton)

In [ ]:
#For numerical features, I use StandardScaler to standardize them.
from sklearn.preprocessing import StandardScaler
def numerical_encoding(df_tmp):
    numerical_columns = df_tmp.select_dtypes(include=['float64', 'int64']).columns
    print(numerical_columns)
    scaler = StandardScaler()

    df1=pd.DataFrame(scaler.fit_transform(df_tmp[numerical_columns]),columns=numerical_columns,index=df_tmp[numerical_columns].index)
    print(df1)
    return df1



In [ ]:
#Apply the function to X_train and X_test
X_train_num_encoded = numerical_encoding(X_train)
X_test_num_encoded = numerical_encoding(X_test)

Index(['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts',
       'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan',
       'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit',
       'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio',
       'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly',
       'Monthly_Balance'],
      dtype='object')
            Age  Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  \
0     -0.786695      -0.830788          -8.182323e-01          -0.907763   
1     -0.786695      -0.830788           3.177067e-16          -0.907763   
2     -2.389605      -0.830788           3.177067e-16          -0.907763   
3     -0.786695      -0.830788           3.177067e-16          -0.907763   
4     -0.786695      -0.830788          -8.182323e-01          -0.907763   
...         ...            ...                    ...                ...   
99995 -0.647312      -0.300500          -2.821

### Concat and Split

In [ ]:
X_train_final = pd.concat([X_train_num_encoded, X_train_cat_encoded], axis=1)
X_test_final = pd.concat([X_test_num_encoded, X_test_cat_encoded], axis=1)

In [ ]:
X_train_final.head()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,Student Loan_True,Mortgage_False,Mortgage_True,Home Equity Loan_False,Home Equity Loan_True,Payday Loan_False,Payday Loan_True,Other Loan_False,Debt Consolidation_False,Debt Consolidation_True
0,-0.786695,-0.830788,-8.182323e-01,-0.907763,-0.753277,-1.29644,0.223368,-1.243146,-0.774561,1.218561e-01,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0
1,-0.786695,-0.830788,3.177067e-16,-0.907763,-0.753277,-1.29644,0.223368,-1.451381,-1.753659,1.218561e-01,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0
2,-2.389605,-0.830788,3.177067e-16,-0.907763,-0.753277,-1.29644,0.223368,-1.243146,-0.774561,2.691210e-16,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0
3,-0.786695,-0.830788,3.177067e-16,-0.907763,-0.753277,-1.29644,0.223368,-1.104323,-1.194175,-6.356522e-01,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0
4,-0.786695,-0.830788,-8.182323e-01,-0.907763,-0.753277,-1.29644,0.223368,-1.034911,-1.753659,1.218561e-01,...,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0


In [ ]:
# Split train data and validation data
X_train, X_val, y_train, y_val = train_test_split(X_train_final, y_train_origin,test_size=0.2, random_state=42)

In [ ]:
X_train_final, y_train_origin

(            Age  Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  \
 0     -0.786695      -0.830788          -8.182323e-01          -0.907763   
 1     -0.786695      -0.830788           3.177067e-16          -0.907763   
 2     -2.389605      -0.830788           3.177067e-16          -0.907763   
 3     -0.786695      -0.830788           3.177067e-16          -0.907763   
 4     -0.786695      -0.830788          -8.182323e-01          -0.907763   
 ...         ...            ...                    ...                ...   
 99995 -0.647312      -0.300500          -2.821720e-01          -0.539980   
 99996 -0.647312      -0.300500          -2.821720e-01          -0.539980   
 99997 -0.647312      -0.300500          -2.821720e-01          -0.539980   
 99998 -0.647312      -0.300500          -2.821720e-01          -0.539980   
 99999 -0.647312      -0.300500          -2.821720e-01          -0.539980   
 
        Num_Credit_Card  Interest_Rate  Num_of_Loan  Delay_from_due_date  

In [ ]:
X_train.shape

(80000, 69)

#Model Training and Model Comparison

##Random Forest

###RF+V0-Feature

#### Model Building

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score
import xgboost as xgb


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Pipeline
pipeline = Pipeline([
    # ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

#### Model Training & Model Evaluation

In [ ]:

scoring = {
    'accuracy': 'accuracy',
    'f1_micro': 'f1_micro',
    'f1_macro': 'f1_macro',
    'precision': 'precision_macro',
    'recall': 'recall_macro'
}

cv_results = cross_validate(
    pipeline, X_train, y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

# Print the average score for each evaluation metric.
for metric in scoring.keys():
    print(f"{metric}: {cv_results['test_' + metric].mean():.4f}")

KeyboardInterrupt: 

In [ ]:
cv_results

#### Error Analysis

In [ ]:
from sklearn.metrics import log_loss, accuracy_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# errors
train_errors = []
val_errors = []

# train_sizes
train_sizes = range(2000, len(X_train),2000)  # 从 1 到 n-1
print(train_sizes)

for n_train in train_sizes:

    X_train1, X_val, y_train1, y_val = train_test_split(X_train, y_train, train_size=n_train, random_state=42, stratify=y_train)
    print(X_train1.shape)
    print(X_val.shape)

    # training
    # model = LogisticRegression(max_iter=50)
    # model.fit(X_train, y_train)
    pipeline.fit(X_train1, y_train1)

    # predict
    y_train_pred_prob =  pipeline.predict_proba(X_train1)
    y_val_pred_prob = pipeline.predict_proba(X_val)

    # log-loss
    train_err = log_loss(y_train1, y_train_pred_prob)
    val_err = log_loss(y_val, y_val_pred_prob)

    train_errors.append(train_err)
    val_errors.append(val_err)

    print(f"Train size: {n_train}, Train log-loss: {train_err:.3f}, Val log-loss: {val_err:.3f}")

# Plotting Learning Curve
plt.plot(train_sizes, train_errors, label='Train log-loss', marker='o')
plt.plot(train_sizes, val_errors, label='Validation log-loss', marker='o')
plt.xlabel("Training set size")
plt.ylabel("Log-loss")
plt.title("Learning Curve - Random Forest")
plt.legend()
plt.show()

### Feature selection + Feature ablation
*  select the Top-K most important features using RandomForest feature_importances_

*  Feature Ablation measures a feature's importance by removing it and observing how the model's performance changes.

In [ ]:
X_train_final.values

In [ ]:
"""
Random Forest -> Feature Importance initial selection -> Feature Ablation (Stratified CV) for 3-class
Optimized: numpy indexing, fold-level parallelism, initial embedded filter to reduce ablation cost.
"""

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.ensemble import RandomForestClassifier
from joblib import Parallel, delayed
import os

# -------------------------
# User-tunable parameters
# -------------------------
RANDOM_SEED = 42
N_SPLITS = 5
AUC_THRESHOLD = 0.001   # acceptance margin for optimal set
N_JOBS = 1              # -1 use all cores, set to 1/2/4 as needed

# Initial embedded selection (fast)
# Options: "top_k" or "cumulative"
INITIAL_SELECT_METHOD = "top_k"  # "top_k" or "cumulative"
INITIAL_TOP_K = 30               # used if method == "top_k"
INITIAL_CUMULATIVE_THRESHOLD = 0.95  # used if method == "cumulative" (e.g. 0.95 -> keep features covering 95% importance)

# RandomForest params for initial importance and ablation CV
RF_IMPORTANCE_PARAMS = {
    "n_estimators": 200,
    "random_state": RANDOM_SEED,
    "n_jobs": N_JOBS
}
RF_ABLATION_PARAMS = {
    "n_estimators": 100,
    "max_depth": None,
    "min_samples_split": 20,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "random_state": RANDOM_SEED,
    "n_jobs": N_JOBS,
}

# -------------------------
# Load data (use user vars or iris demo)
# -------------------------

X = X_train_final  # try alternate name
y = y_train_origin
print("Using X_train_final / y_train_origin from environment.")

feature_names = X.columns.tolist()
n_classes = len(np.unique(y))
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}, Classes: {n_classes}")

# binarize labels for ROC AUC multiclass OvR
y_bin = label_binarize(y, classes=np.arange(n_classes))

# convert to numpy for speed
X_values = X.values
y_values = y

# -------------------------
# 1) Train RF to get feature_importances_ (fast embedded filter)
# -------------------------
t0 = time.time()
rf_importance = RandomForestClassifier(**RF_IMPORTANCE_PARAMS)
rf_importance.fit(X_values, y_values)
importances = rf_importance.feature_importances_  # shape (n_features,)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False).reset_index(drop=True)

# select initial set
if INITIAL_SELECT_METHOD == "top_k":
    top_k = min(INITIAL_TOP_K, X.shape[1])
    selected_features = importance_df['feature'].iloc[:top_k].tolist()
    print(f"Initial selection: top_k = {top_k} features")
elif INITIAL_SELECT_METHOD == "cumulative":
    cum = importance_df['importance'].cumsum()
    mask = cum <= INITIAL_CUMULATIVE_THRESHOLD
    # ensure at least 1 feature is selected
    if mask.sum() == 0:
        mask.iloc[0] = True
    selected_features = importance_df.loc[mask, 'feature'].tolist()
    print(f"Initial selection: cumulative threshold = {INITIAL_CUMULATIVE_THRESHOLD}, selected {len(selected_features)} features")
else:
    selected_features = importance_df['feature'].tolist()
    print("Initial selection: using all features (no filtering)")

t1 = time.time()
print(f"[Importance] computed and initial filtered in {t1 - t0:.2f}s")
print("Selected features (initial):", selected_features[:50])

# -------------------------
# 2) Feature Ablation (iterative removal of least important among selected_features)
#    Use StratifiedKFold CV (ROC AUC OvR macro) and parallel folds
# -------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
splits = list(skf.split(X_values, y_values))

# Map feature->col idx for fast numpy slicing
feat_to_idx = {f: i for i, f in enumerate(feature_names)}
remaining_idxs = [feat_to_idx[f] for f in selected_features]

auc_records = []

def train_rf_and_auc_np(X_tr_np, y_tr_np, X_val_np, y_val_bin_np):
    """Train RF on numpy arrays and compute multiclass ROC AUC (OvR macro)."""
    model = RandomForestClassifier(**RF_ABLATION_PARAMS)
    model.fit(X_tr_np, y_tr_np)
    y_pred_proba = model.predict_proba(X_val_np)
    auc = roc_auc_score(y_val_bin_np, y_pred_proba, average='macro', multi_class='ovr')
    return auc

t_ab0 = time.time()
total_iters = len(remaining_idxs)
for it in range(total_iters):
    # remove least-important feature among current remaining (we use importance_df order as base)
    # find current remaining features' importance to decide removal order
    cur_features = [feature_names[idx] for idx in remaining_idxs]
    # get importances in descending order for current features (recompute from previous importance ranking)
    cur_importance_order = sorted(cur_features, key=lambda f: importance_df[importance_df['feature']==f]['importance'].values[0], reverse=True)
    # We'll remove the last (least important) in that order
    if len(cur_importance_order) > 1:
        removed_feature = cur_importance_order[-1]
        removed_idx = feat_to_idx[removed_feature]
        # remove that idx from remaining_idxs
        remaining_idxs = [i for i in remaining_idxs if i != removed_idx]
    else:
        removed_feature = None

    cur_idxs = np.array(remaining_idxs, dtype=int)
    # define fold computation
    def _fold_auc(train_idx, val_idx):
        X_tr = X_values[train_idx][:, cur_idxs]
        X_val = X_values[val_idx][:, cur_idxs]
        y_tr = y_values[train_idx]
        y_val_bin = y_bin[val_idx]
        return train_rf_and_auc_np(X_tr, y_tr, X_val, y_val_bin)

    # parallel compute over folds
    fold_aucs = Parallel(n_jobs=N_JOBS)(
        delayed(_fold_auc)(tr, va) for tr, va in splits
    )

    mean_auc = float(np.mean(fold_aucs))
    auc_records.append({
        "num_features": len(cur_idxs),
        "removed_feature": removed_feature,
        "mean_cv_auc": mean_auc,
        "remaining_features": [feature_names[idx] for idx in cur_idxs]
    })

    print(f"[Iter {it+1}/{total_iters}] removed={removed_feature}, num_features={len(cur_idxs)}, mean_cv_auc={mean_auc:.4f}")

t_ab1 = time.time()
print(f"[Ablation] Finished {total_iters} iterations in {t_ab1 - t_ab0:.1f}s")

ablation_df = pd.DataFrame(auc_records)

# -------------------------
# 3) Choose optimal subset: first subset with mean_cv_auc >= max_auc - AUC_THRESHOLD
# -------------------------
max_auc = ablation_df['mean_cv_auc'].max()
optimal_row = ablation_df[ablation_df['mean_cv_auc'] >= (max_auc - AUC_THRESHOLD)].iloc[0]
optimal_features = optimal_row['remaining_features']
print(f"Max mean CV AUC: {max_auc:.4f}")
print(f"Optimal features ({len(optimal_features)}): {optimal_features}")


# -------------------------
# 5) Save & plot
# -------------------------
out_dir = "./rf_feature_ablation_output"
os.makedirs(out_dir, exist_ok=True)
ablation_df.to_csv(os.path.join(out_dir, "ablation_results.csv"), index=False)
pd.DataFrame({'optimal_features': optimal_features}).to_csv(os.path.join(out_dir, "optimal_features.csv"), index=False)
print(f"Saved results to {out_dir}")

# Plot ablation curve
plt.figure(figsize=(8,5))
plt.plot(ablation_df['num_features'], ablation_df['mean_cv_auc'], marker='o')
plt.axhline(max_auc - AUC_THRESHOLD, color='red', linestyle='--', label='AUC threshold')
plt.gca().invert_xaxis()
plt.xlabel("Number of features")
plt.ylabel("Mean CV ROC AUC (macro, OvR)")
plt.title("Feature Ablation Curve (RF importance initial filter + Ablation)")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

# -------------------------
# 4) Final: train RF on optimal_features with larger n_estimators (optional)
# -------------------------
FINAL_RF_PARAMS = {
    "n_estimators": 500,
    "max_depth": None,
    "random_state": RANDOM_SEED,
    "n_jobs": N_JOBS
}
final_rf = RandomForestClassifier(**FINAL_RF_PARAMS)
final_rf.fit(X[optimal_features].values, y_values)
print("Trained final RF on optimal features with n_estimators =", FINAL_RF_PARAMS["n_estimators"])

### Super parameters adjusting

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np

# 1) Load the list of optimal features
try:
    RF_OPTIMAL_FEATURES = list(optimal_features)
    print(f"Loaded {len(RF_OPTIMAL_FEATURES)} optimal features from ablation block.")
except NameError:
    RF_OPTIMAL_FEATURES = [
        'Outstanding_Debt', 'Interest_Rate', 'Delay_from_due_date',
        'Changed_Credit_Limit', 'Credit_History_Age', 'Num_Credit_Inquiries',
        'Num_Credit_Card', 'Annual_Income', 'Amount_invested_monthly',
        'Total_EMI_per_month', 'Monthly_Inhand_Salary', 'Age',
        'Num_of_Delayed_Payment', 'Credit_Utilization_Ratio',
        'Num_Bank_Accounts', 'Credit_Mix_Good', 'Credit_Mix_Standard',
        'Num_of_Loan', 'Payment_of_Min_Amount_Yes'
    ]
    print("optimal_features variable not found; using fallback list.")

# 2) Prepare training and validation data
if 'X_train' in globals() and 'y_train' in globals():
    print("Using split training set (X_train / y_train) for hyperparameter tuning.")
    X_rf_train = X_train[RF_OPTIMAL_FEATURES].copy()
    y_rf_train = y_train
else:
    print("Warning: X_train / y_train not found! Using X_train_final / y_train_origin as fallback.")
    X_rf_train = X_train_final[RF_OPTIMAL_FEATURES].copy()
    y_rf_train = y_train_origin

# 3) Define model and parameter space
base_rf = RandomForestClassifier(
    random_state=RANDOM_SEED,
    n_jobs=N_JOBS
)

param_grid_rf = {
    "n_estimators": [200, 400],
    "max_depth": [10, 20, None],
    "min_samples_split": [5, 10],
    "min_samples_leaf": [2, 4],
    "max_features": ["sqrt"]
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

search_rf = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_grid_rf,
    n_iter=20,
    scoring="roc_auc_ovr",
    cv=skf,
    n_jobs=N_JOBS,
    verbose=2,
    random_state=RANDOM_SEED
)

# 4) Hyperparameter tuning
print(f"\n[RF] Starting Hyperparameter Tuning on {X_rf_train.shape} ...")
search_rf.fit(X_rf_train, y_rf_train)

print("\n================ RF Tuning Result ================")
print(f"Best CV macro ROC-AUC: {search_rf.best_score_:.4f}")
print("Best params:", search_rf.best_params_)

best_rf = search_rf.best_estimator_

# 5) Evaluate on validation set
if 'X_val' in globals() and 'y_val' in globals():
    print("\n[RF] Evaluating on Validation Set (strictly unseen data)...")
    X_val_rf = X_val[RF_OPTIMAL_FEATURES].copy()

    y_val_pred = best_rf.predict(X_val_rf)
    y_val_proba = best_rf.predict_proba(X_val_rf)

    print(classification_report(y_val, y_val_pred))

    try:
        val_auc = roc_auc_score(y_val, y_val_proba, multi_class="ovr", average="macro")
        print(f"Validation Macro ROC-AUC: {val_auc:.4f}")
    except Exception as e:
        print("AUC Calculation Error:", e)

    cm = confusion_matrix(y_val, y_val_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(xticks_rotation=45)
    plt.title("Optimized Random Forest Confusion Matrix")
    plt.show()

# 6) Retrain final model on the full dataset (train + validation)
print("\n[RF] Retraining final model on the entire training data (X_train_final / y_train_origin)...")
X_full = X_train_final[RF_OPTIMAL_FEATURES].copy()
y_full = y_train_origin

final_rf = RandomForestClassifier(**search_rf.best_params_, random_state=RANDOM_SEED, n_jobs=N_JOBS)
final_rf.fit(X_full, y_full)

print("[RF] Final model retrained successfully on full data.")


### Error Analysis

In [ ]:
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# 1) Define the tuned Random Forest model (parameters obtained from tuning)
best_rf = RandomForestClassifier(
    n_estimators=400,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    max_depth=None,
    random_state=42,
    n_jobs=1
)

# 2) Initialize lists to record training and validation errors
train_errors = []
val_errors = []

# 3) Define the range of training set sizes (increase by 2000 each time)
train_sizes = range(2000, len(X_train_final), 2000)
print("Training sizes:", list(train_sizes))

# 4) Iteratively train and evaluate model on increasing sample sizes
for n_train in train_sizes:
    # Split the full training data into a random train/validation set for the current size
    X_train1, X_val, y_train1, y_val = train_test_split(
        X_train_final,
        y_train_origin,
        train_size=n_train,
        random_state=42,
        stratify=y_train_origin
    )

    print(f"\nTraining with {n_train} samples...")
    print("X_train1 shape:", X_train1.shape)
    print("X_val shape:", X_val.shape)

    # Train the model on the current subset
    best_rf.fit(X_train1, y_train1)

    # Predict probabilities on both training and validation sets
    y_train_pred_prob = best_rf.predict_proba(X_train1)
    y_val_pred_prob = best_rf.predict_proba(X_val)

    # Compute log-loss for both sets
    train_err = log_loss(y_train1, y_train_pred_prob)
    val_err = log_loss(y_val, y_val_pred_prob)

    # Record the results
    train_errors.append(train_err)
    val_errors.append(val_err)

    print(f"Train size: {n_train}, Train log-loss: {train_err:.3f}, Val log-loss: {val_err:.3f}")

# 5) Plot the learning curve
plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_errors, label='Train log-loss', marker='o')
plt.plot(train_sizes, val_errors, label='Validation log-loss', marker='o')
plt.xlabel("Training set size")
plt.ylabel("Log-loss")
plt.title("Learning Curve - Tuned Random Forest Model")
plt.legend()
plt.grid(True)
plt.show()


## Xgboost

###XGB+ V0-Feature

In [ ]:
# Split train data and validation data
X_train, X_val, y_train, y_val

(            Age  Annual_Income  Monthly_Inhand_Salary  Num_Bank_Accounts  \
 75220 -0.995771       1.296284           1.526099e+00          -1.643331   
 48955 -0.647312       1.593120           1.927457e+00          -0.172196   
 44966  1.025289      -0.598474          -6.964514e-01          -0.172196   
 13568 -0.020087      -0.744862          -8.070496e-01           0.563371   
 92727 -0.438237      -0.782681          -7.766257e-01          -1.275547   
 ...         ...            ...                    ...                ...   
 6265   0.816214       0.796499           9.098368e-01          -0.172196   
 54886 -0.995771      -0.939203          -9.581111e-01           0.195587   
 76820  0.258680       0.403397           5.600811e-01          -0.907763   
 860    1.443439      -0.823317          -7.902491e-01          -1.643331   
 15795 -0.089778      -0.829315           3.177067e-16           1.666722   
 
        Num_Credit_Card  Interest_Rate  Num_of_Loan  Delay_from_due_date  

In [ ]:
optimal_features = [
    'Credit_Mix_Good',
    'Credit_Mix_Standard',
    'Payment_of_Min_Amount_No',
    'Credit_Mix_unknown_credit_mix',
    'Outstanding_Debt',
    'Credit_Mix_Bad',
    'Interest_Rate',
    'Month_February',
    'Month_January',
    'Num_Credit_Card',
    'Month_March',
    'Delay_from_due_date',
    'Num_Bank_Accounts',
    'Changed_Credit_Limit',
    'Payment_Behaviour_Low_spent_Small_value_payments',
    'Total_EMI_per_month',
    'Num_Credit_Inquiries',
    'Month_July',
    'Credit-Builder Loan_False',
    'Annual_Income',
    'Num_of_Loan',
    'Occupation_Entrepreneur',
    'Num_of_Delayed_Payment',
    'Month_August',
    'Occupation_Engineer',
    'Personal Loan_False',
    'Debt Consolidation_False',
    'Credit_History_Age',
    'Occupation_Writer'
]

# Subset train/validation/full/train-test matrices to the optimal features
X_train_opt = X_train[optimal_features].copy()
X_val_opt   = X_val[optimal_features].copy()

In [ ]:
X_train_opt

,Credit_Mix_Good,Credit_Mix_Standard,Payment_of_Min_Amount_No,Credit_Mix_unknown_credit_mix,Outstanding_Debt,Credit_Mix_Bad,Interest_Rate,Month_February,Month_January,Num_Credit_Card,...,Annual_Income,Num_of_Loan,Occupation_Entrepreneur,Num_of_Delayed_Payment,Month_August,Occupation_Engineer,Personal Loan_False,Debt Consolidation_False,Credit_History_Age,Occupation_Writer
75220,1.0,0.0,1.0,0.0,-0.316122,0.0,-1.404396,0.0,0.0,-1.207751,...,1.296284,-0.563180,0.0,-0.914432,0.0,0.0,0.0,1.0,0.837810,0.0
48955,1.0,0.0,1.0,0.0,-0.608870,0.0,-0.540747,0.0,0.0,-2.116699,...,1.593120,0.223368,0.0,-0.354948,0.0,0.0,1.0,0.0,1.275178,0.0
44966,0.0,1.0,1.0,0.0,-0.841079,0.0,-0.756659,0.0,0.0,0.610145,...,-0.598474,-0.169906,0.0,-0.634690,0.0,0.0,1.0,0.0,1.047747,0.0
13568,0.0,0.0,1.0,1.0,-1.089624,0.0,-0.648703,0.0,1.0,0.610145,...,-0.744862,0.223368,0.0,0.064665,0.0,0.0,1.0,0.0,-1.760158,0.0
92727,1.0,0.0,1.0,0.0,-0.601512,0.0,-0.648703,0.0,0.0,-1.207751,...,-0.782681,-0.956453,0.0,-0.494819,1.0,0.0,1.0,1.0,0.785326,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6265,1.0,0.0,1.0,0.0,-1.142207,0.0,-1.296440,1.0,0.0,-0.298803,...,0.796499,-0.169906,0.0,-0.494819,0.0,0.0,1.0,1.0,0.654115,0.0
54886,0.0,1.0,0.0,0.0,-0.119156,0.0,0.106990,0.0,0.0,1.973567,...,-0.939203,-0.563180,0.0,0.484278,0.0,0.0,1.0,1.0,-0.351832,0.0
76820,0.0,1.0,1.0,0.0,-0.713898,0.0,2.482026,0.0,0.0,0.155671,...,0.403397,0.223368,0.0,-0.354948,0.0,0.0,1.0,1.0,1.187705,0.0
860,0.0,0.0,1.0,1.0,-0.941121,0.0,-1.296440,0.0,0.0,0.610145,...,-0.823317,-0.169906,0.0,-0.494819,0.0,0.0,1.0,0.0,-0.010685,0.0


#### Model Building

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score
import xgboost as xgb

optimal_features = [
    'Credit_Mix_Good',
    'Credit_Mix_Standard',
    'Payment_of_Min_Amount_No',
    'Credit_Mix_unknown_credit_mix',
    'Outstanding_Debt',
    'Credit_Mix_Bad',
    'Interest_Rate',
    'Month_February',
    'Month_January',
    'Num_Credit_Card',
    'Month_March',
    'Delay_from_due_date',
    'Num_Bank_Accounts',
    'Changed_Credit_Limit',
    'Payment_Behaviour_Low_spent_Small_value_payments',
    'Total_EMI_per_month',
    'Num_Credit_Inquiries',
    'Month_July',
    'Credit-Builder Loan_False',
    'Annual_Income',
    'Num_of_Loan',
    'Occupation_Entrepreneur',
    'Num_of_Delayed_Payment',
    'Month_August',
    'Occupation_Engineer',
    'Personal Loan_False',
    'Debt Consolidation_False',
    'Credit_History_Age',
    'Occupation_Writer'
]

# Subset train/validation/full/train-test matrices to the optimal features
X_train_opt = X_train[optimal_features].copy()
X_val_opt   = X_val[optimal_features].copy()


print("X_train_opt shape:", X_train_opt.shape)
print("X_val_opt shape:", X_val_opt.shape)

num_classes = len(np.unique(y_train))
# --- 2. 实例化并训练模型 ---
xgb_model = xgb.XGBClassifier(

  n_estimators=589,
  max_depth=10,
  learning_rate=0.1284132359495751,
  subsample=0.8873103693742401,
  colsample_bytree=0.9397409133893042,
  min_child_weight=6,
  gamma=0.027716566946387168,
  reg_lambda=0.05606118505874246,
  reg_alpha=0.6911098857185791,
    objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
        n_jobs= -1

)

print("开始训练 XGBoost 模型...")
xgb_model.fit(X_train_opt, y_train)
print("模型训练完成。")

# 简单评估一下
y_pred = xgb_model.predict(X_val_opt)
print(f"测试集准确率: {accuracy_score(y_val, y_pred):.4f}")



y_val_pred  = xgb_model.predict(X_val_opt)
y_val_proba = xgb_model.predict_proba(X_val_opt)

print("\nValidation classification report:")
print(classification_report(y_val, y_val_pred))

print("Validation confusion matrix:")
print(confusion_matrix(y_val, y_val_pred))

val_auc = roc_auc_score(y_val, y_val_proba, multi_class="ovr")
print("\nValidation macro ROC-AUC (Optuna best params): {:.4f}".format(val_auc))

X_train_opt shape: (80000, 29)
X_val_opt shape: (20000, 29)
开始训练 XGBoost 模型...
模型训练完成。
测试集准确率: 0.8116

Validation classification report:
              precision    recall  f1-score   support

           0       0.78      0.76      0.77      3527
           1       0.81      0.82      0.81      5874
           2       0.82      0.82      0.82     10599

    accuracy                           0.81     20000
   macro avg       0.80      0.80      0.80     20000
weighted avg       0.81      0.81      0.81     20000

Validation confusion matrix:
[[2682    7  838]
 [  37 4816 1021]
 [ 716 1149 8734]]

Validation macro ROC-AUC (Optuna best params): 0.9304


In [ ]:

# 简单评估一下
print("训练集效果")
y_pred_train = xgb_model.predict(X_train_opt)
print(f"训练集集准确率: {accuracy_score(y_train, y_pred_train):.4f}")

y_train_prob = xgb_model.predict_proba(X_train_opt)

print("\nTrain classification report:")
print(classification_report(y_train, y_pred_train))

print("Train confusion matrix:")
print(confusion_matrix(y_train, y_pred_train))

val_auc = roc_auc_score(y_train, y_train_prob, multi_class="ovr")
print("\nTrain macro ROC-AUC (Optuna best params): {:.4f}".format(val_auc))

训练集效果
训练集集准确率: 0.9963

Train classification report:
              precision    recall  f1-score   support

           0       0.99      1.00      1.00     14301
           1       0.99      1.00      1.00     23124
           2       1.00      0.99      1.00     42575

    accuracy                           1.00     80000
   macro avg       1.00      1.00      1.00     80000
weighted avg       1.00      1.00      1.00     80000

Train confusion matrix:
[[14291     0    10]
 [    0 23093    31]
 [   87   168 42320]]

Train macro ROC-AUC (Optuna best params): 1.0000


In [ ]:
# --- 3B. 保存模型 (Joblib) ---
import joblib
filename_joblib = 'xgboost_model_joblib.pkl'

# 使用 joblib.dump 保存
joblib.dump(xgb_model, filename_joblib)

print(f"\n模型已使用 joblib 保存到：{filename_joblib}")

# --- 4B. 加载模型 (Joblib) ---
loaded_model_joblib = joblib.load(filename_joblib)

print("模型已成功加载 (Joblib)。")


模型已使用 joblib 保存到：xgboost_model_joblib.pkl
模型已成功加载 (Joblib)。


In [ ]:
LOCAL_FILENAME = 'my_xgb_model_for_download.joblib'
joblib.dump(xgb_model, LOCAL_FILENAME)

['my_xgb_model_for_download.joblib']

In [ ]:
from google.colab import files
files.download(LOCAL_FILENAME)

print(f"文件 {LOCAL_FILENAME} 已生成，请查看您的浏览器下载。")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

文件 my_xgb_model_for_download.joblib 已生成，请查看您的浏览器下载。


In [ ]:
loaded_model_joblib.predict(X_train_opt)

array([2, 0, 0, ..., 2, 2, 1])

In [ ]:



skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Pipeline with XGBoost
pipeline = Pipeline([
    # ('scaler', StandardScaler()),  # 如果需要，可以取消注释
    ('xgb', xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,          # XGBoost 默认6，你可以调整
        learning_rate=0.1,    # 学习率
        subsample=0.8,        # 样本采样率
        colsample_bytree=0.8, # 特征采样率
        reg_lambda=1,         # L2 正则化
        reg_alpha=0,          # L1 正则化
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1
    ))
])


#### Model Training & Model Evaluation

In [ ]:

scoring = {
    'accuracy': 'accuracy',
    'f1_micro': 'f1_micro',
    'f1_macro': 'f1_macro',
    'precision': 'precision_macro',
    'recall': 'recall_macro'
}

cv_results = cross_validate(
    pipeline, X_train_final, y_train_origin,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

# Print the average score for each evaluation metric.
for metric in scoring.keys():
    print(f"{metric}: {cv_results['test_' + metric].mean():.4f}")

In [ ]:
cv_results

#### Error Analysis

In [ ]:
from sklearn.metrics import log_loss, accuracy_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# errors
train_errors = []
val_errors = []

# train_sizes
train_sizes = range(2000, len(X_train),2000)  # 从 1 到 n-1
print(train_sizes)

for n_train in train_sizes:

    X_train1, X_val, y_train1, y_val = train_test_split(X_train, y_train, train_size=n_train, random_state=42, stratify=y_train)
    print(X_train1.shape)
    print(X_val.shape)

    # training
    # model = LogisticRegression(max_iter=50)
    # model.fit(X_train, y_train)
    pipeline.fit(X_train1, y_train1)

    # predict
    y_train_pred_prob =  pipeline.predict_proba(X_train1)
    y_val_pred_prob = pipeline.predict_proba(X_val)

    # log-loss
    train_err = log_loss(y_train1, y_train_pred_prob)
    val_err = log_loss(y_val, y_val_pred_prob)

    train_errors.append(train_err)
    val_errors.append(val_err)

    print(f"Train size: {n_train}, Train log-loss: {train_err:.3f}, Val log-loss: {val_err:.3f}")

# Plotting Learning Curve
plt.plot(train_sizes, train_errors, label='Train log-loss', marker='o')
plt.plot(train_sizes, val_errors, label='Validation log-loss', marker='o')
plt.xlabel("Training set size")
plt.ylabel("Log-loss")
plt.title("Learning Curve - Random Forest")
plt.legend()
plt.show()

###Feature slection + Feature Ablation


In [ ]:
"""
XGBoost -> Importance (gain) initial selection -> Feature Ablation (Stratified CV) for 3-class
Fast, practical pipeline. Adjust parameters at top as needed.
"""
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from joblib import Parallel, delayed
from xgboost import XGBClassifier

# -------------------------
# User-tunable parameters
# -------------------------
RANDOM_SEED = 42
N_SPLITS = 5
AUC_THRESHOLD = 0.001   # tolerance to select first subset near max AUC
N_JOBS = 1             # -1 uses all available threads

# Initial selection: choose one
INITIAL_SELECT_METHOD = "top_k"  # "top_k" or "cumulative"
INITIAL_TOP_K = 30                    # if method=="top_k"
INITIAL_CUMULATIVE_THRESHOLD = 0.95    # if method=="cumulative" (e.g. keep features covering 95% gain)

# XGBoost params used for importance calculation (fast)
XGB_IMPORTANCE_PARAMS = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.1,
    "objective": "multi:softprob",
    "verbosity": 0,
    "random_state": RANDOM_SEED,
    "use_label_encoder": False,
    "n_jobs": N_JOBS
}

# XGBoost params used for ablation CV (faster, moderate)
XGB_ABLATION_PARAMS = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "objective": "multi:softprob",
    "verbosity": 0,
    "random_state": RANDOM_SEED,
    "use_label_encoder": False,
    "n_jobs": N_JOBS
}

# Final model params (optional re-train)
XGB_FINAL_PARAMS = {
    "n_estimators": 800,
    "max_depth": 6,
    "learning_rate": 0.05,
    "objective": "multi:softprob",
    "verbosity": 0,
    "random_state": RANDOM_SEED,
    "use_label_encoder": False,
    "n_jobs": N_JOBS
}

# -------------------------
# Load input data (use environment or fallback)
# -------------------------
X = X_train_final  # if user has provided these names
y = y_train_origin
print("Using X_train_final / y_train_origin from environment.")

feature_names = X.columns.tolist()
n_classes = len(np.unique(y))
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}, Classes: {n_classes}")

# Binarize for multiclass ROC AUC (OvR macro)
y_bin = label_binarize(y, classes=np.arange(n_classes))

# Convert to numpy for speed (we will keep pandas for XGBoost fit to preserve feature names)
X_values = X.values
y_values = y

# -------------------------
# 1) Train XGBoost to get feature importances (gain)
# -------------------------
t0 = time.time()
xgb_imp = XGBClassifier(**XGB_IMPORTANCE_PARAMS)
# Fit using pandas DataFrame to preserve column names in booster.get_score
xgb_imp.fit(X, y)
# Use booster.get_score with importance_type='gain'
importance_dict = xgb_imp.get_booster().get_score(importance_type="gain")  # keys like feature names

# Map any missing features to importance 0
all_importances = []
for f in feature_names:
    all_importances.append(importance_dict.get(f, 0.0))

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": np.array(all_importances, dtype=float)
}).sort_values("importance", ascending=False).reset_index(drop=True)

# Normalize to relative importance (sum to 1) for cumulative filtering
if importance_df["importance"].sum() > 0:
    importance_df["importance_norm"] = importance_df["importance"] / importance_df["importance"].sum()
else:
    importance_df["importance_norm"] = importance_df["importance"]

# Choose initial set
if INITIAL_SELECT_METHOD == "top_k":
    top_k = min(INITIAL_TOP_K, X.shape[1])
    selected_features = importance_df["feature"].iloc[:top_k].tolist()
    print(f"Initial selection: top_k = {top_k} features")
elif INITIAL_SELECT_METHOD == "cumulative":
    importance_df["cum_importance"] = importance_df["importance_norm"].cumsum()
    mask = importance_df["cum_importance"] <= INITIAL_CUMULATIVE_THRESHOLD
    # ensure at least 1 feature selected
    if mask.sum() == 0:
        mask.iloc[0] = True
    selected_features = importance_df.loc[mask, "feature"].tolist()
    print(f"Initial selection: cumulative threshold = {INITIAL_CUMULATIVE_THRESHOLD}, selected {len(selected_features)} features")
else:
    selected_features = importance_df["feature"].tolist()
    print("Initial selection: using all features (no filter)")

t1 = time.time()
print(f"[Importance] computed and initial filtered in {t1 - t0:.2f}s")
print("Top selected features (initial):", selected_features[:50])

# -------------------------
# 2) Feature Ablation (iteratively remove least-important among selected_features)
#    Evaluate with Stratified K-Fold CV using ROC AUC OvR macro
# -------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
splits = list(skf.split(X_values, y_values))  # precompute folds (index pairs)

# Map feature -> column idx for fast slicing
feat_to_idx = {f: i for i, f in enumerate(feature_names)}
remaining_idxs = [feat_to_idx[f] for f in selected_features]

auc_records = []

def train_xgb_and_auc_np(X_tr_np, y_tr_np, X_val_np, y_val_bin_np):
    """Train XGBoost on numpy arrays and compute multiclass ROC AUC (OvR macro)."""
    # instantiate a fresh model for each fold
    model = XGBClassifier(**XGB_ABLATION_PARAMS)
    # fit accepts numpy arrays
    model.fit(X_tr_np, y_tr_np)
    y_pred_proba = model.predict_proba(X_val_np)
    auc = roc_auc_score(y_val_bin_np, y_pred_proba, average="macro", multi_class="ovr")
    return auc

t_ab0 = time.time()
total_iters = len(remaining_idxs)
for it in range(total_iters):
    # choose least-important to remove based on original importance order (importance_df)
    cur_features = [feature_names[idx] for idx in remaining_idxs]
    # order by importance_df
    cur_order = sorted(cur_features, key=lambda f: importance_df[importance_df["feature"] == f]["importance"].values[0], reverse=True)
    if len(cur_order) > 1:
        removed_feature = cur_order[-1]
        removed_idx = feat_to_idx[removed_feature]
        # update remaining_idxs by removing removed_idx
        remaining_idxs = [i for i in remaining_idxs if i != removed_idx]
    else:
        removed_feature = None

    cur_idxs = np.array(remaining_idxs, dtype=int)

    # fold function
    def _fold_auc(train_idx, val_idx):
        X_tr = X_values[train_idx][:, cur_idxs]
        X_val = X_values[val_idx][:, cur_idxs]
        y_tr = y_values[train_idx]
        y_val_bin = y_bin[val_idx]
        return train_xgb_and_auc_np(X_tr, y_tr, X_val, y_val_bin)

    # compute folds in parallel
    fold_aucs = Parallel(n_jobs=N_JOBS)(
        delayed(_fold_auc)(tr, va) for tr, va in splits
    )

    mean_auc = float(np.mean(fold_aucs))
    auc_records.append({
        "num_features": len(cur_idxs),
        "removed_feature": removed_feature,
        "mean_cv_auc": mean_auc,
        "remaining_features": [feature_names[idx] for idx in cur_idxs]
    })

    print(f"Iter {it+1}/{total_iters}: removed={removed_feature}, #features={len(cur_idxs)}, mean_auc={mean_auc:.4f}")

t_ab1 = time.time()
print(f"[Ablation] Finished {total_iters} iterations in {t_ab1 - t_ab0:.1f}s")

ablation_df = pd.DataFrame(auc_records)

# -------------------------
# 3) Pick optimal subset: first subset with mean_cv_auc >= max_auc - AUC_THRESHOLD
# -------------------------
max_auc = ablation_df["mean_cv_auc"].max()
optimal_row = ablation_df[ablation_df["mean_cv_auc"] >= (max_auc - AUC_THRESHOLD)].iloc[0]
optimal_features = optimal_row["remaining_features"]

print(f"Max mean CV AUC: {max_auc:.4f}")
print(f"Optimal feature set ({len(optimal_features)} features): {optimal_features}")



# -------------------------
# 5) Save and plot
# -------------------------
out_dir = "./xgb_feature_ablation_output"
os.makedirs(out_dir, exist_ok=True)
ablation_df.to_csv(os.path.join(out_dir, "ablation_results.csv"), index=False)
pd.DataFrame({"optimal_features": optimal_features}).to_csv(os.path.join(out_dir, "optimal_features.csv"), index=False)
importance_df.to_csv(os.path.join(out_dir, "importance_df.csv"), index=False)
print("Saved results to", out_dir)

# Plot ablation curve
plt.figure(figsize=(8,5))
plt.plot(ablation_df["num_features"], ablation_df["mean_cv_auc"], marker="o")
plt.axhline(max_auc - AUC_THRESHOLD, color="red", linestyle="--", label="AUC threshold")
plt.gca().invert_xaxis()
plt.xlabel("Number of features")
plt.ylabel("Mean CV ROC AUC (macro, OvR)")
plt.title("Feature Ablation Curve (XGBoost importance initial filter + Ablation)")
plt.legend()
plt.grid(True)
plt.show()

# Quick importance bar (top 30)
plt.figure(figsize=(8,6))
plot_df = importance_df.head(30).iloc[::-1]
plt.barh(plot_df["feature"], plot_df["importance"])
plt.xlabel("Gain importance")
plt.title("Top 30 XGBoost feature importances (gain)")
plt.tight_layout()
plt.show()


In [ ]:
# -------------------------
# 4) Optional: retrain final XGBoost on optimal features with larger n_estimators
# -------------------------
final_model = XGBClassifier(**XGB_FINAL_PARAMS)
final_model.fit(X[optimal_features], y)
print("Trained final XGBoost on optimal features with params:", XGB_FINAL_PARAMS)

### Super parameters adjusting （xiaoxi）

In [ ]:
!pip install optuna

import numpy as np
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Optimal feature set from previous XGBoost feature selection
optimal_features = [
    'Credit_Mix_Good',
    'Credit_Mix_Standard',
    'Payment_of_Min_Amount_No',
    'Credit_Mix_unknown_credit_mix',
    'Outstanding_Debt',
    'Credit_Mix_Bad',
    'Interest_Rate',
    'Month_February',
    'Month_January',
    'Num_Credit_Card',
    'Month_March',
    'Delay_from_due_date',
    'Num_Bank_Accounts',
    'Changed_Credit_Limit',
    'Payment_Behaviour_Low_spent_Small_value_payments',
    'Total_EMI_per_month',
    'Num_Credit_Inquiries',
    'Month_July',
    'Credit-Builder Loan_False',
    'Annual_Income',
    'Num_of_Loan',
    'Occupation_Entrepreneur',
    'Num_of_Delayed_Payment',
    'Month_August',
    'Occupation_Engineer',
    'Personal Loan_False',
    'Debt Consolidation_False',
    'Credit_History_Age',
    'Occupation_Writer'
]

# Subset train/validation/full/train-test matrices to the optimal features
X_train_opt = X_train[optimal_features].copy()
X_val_opt   = X_val[optimal_features].copy()
X_full_opt  = X_train_final[optimal_features].copy()
X_test_opt  = X_test_final[optimal_features].copy()

print("X_train_opt shape:", X_train_opt.shape)
print("X_val_opt shape:", X_val_opt.shape)

num_classes = len(np.unique(y_train))

# Objective function for Optuna
def objective(trial: optuna.trial.Trial) -> float:
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 0.4),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
        "objective": "multi:softprob",
        "num_class": num_classes,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    }

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_opt, y_train):
        X_tr, X_va = X_train_opt.iloc[train_idx], X_train_opt.iloc[valid_idx]
        y_tr, y_va = np.array(y_train)[train_idx], np.array(y_train)[valid_idx]

        model = XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)

        y_proba = model.predict_proba(X_va)
        auc = roc_auc_score(y_va, y_proba, multi_class="ovr")
        scores.append(auc)

    return float(np.mean(scores))

# Run Optuna optimization
study = optuna.create_study(direction="maximize", study_name="xgb_optimal_features")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Number of finished trials:", len(study.trials))
print("Best trial AUC:", study.best_value)
print("Best trial params:")
for k, v in study.best_trial.params.items():
    print(f"  {k}: {v}")

# Train best model on training set and evaluate on validation set
best_params = study.best_trial.params
best_params.update({
    "objective": "multi:softprob",
    "num_class": num_classes,
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1
})

best_xgb = XGBClassifier(**best_params)
best_xgb.fit(X_train_opt, y_train)

y_val_pred  = best_xgb.predict(X_val_opt)
y_val_proba = best_xgb.predict_proba(X_val_opt)

print("\nValidation classification report:")
print(classification_report(y_val, y_val_pred))

print("Validation confusion matrix:")
print(confusion_matrix(y_val, y_val_pred))

val_auc = roc_auc_score(y_val, y_val_proba, multi_class="ovr")
print("\nValidation macro ROC-AUC (Optuna best params): {:.4f}".format(val_auc))


### Evaluation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize

# ----- Basic metrics on train and validation -----
y_train_pred = best_xgb.predict(X_train_opt)
y_train_proba = best_xgb.predict_proba(X_train_opt)

y_val_pred = best_xgb.predict(X_val_opt)
y_val_proba = best_xgb.predict_proba(X_val_opt)

train_acc = accuracy_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred, average="macro")
train_auc = roc_auc_score(y_train, y_train_proba, multi_class="ovr")

val_acc = accuracy_score(y_val, y_val_pred)
val_f1 = f1_score(y_val, y_val_pred, average="macro")
val_auc = roc_auc_score(y_val, y_val_proba, multi_class="ovr")

print("Train accuracy:", train_acc)
print("Train macro F1:", train_f1)
print("Train macro ROC-AUC:", train_auc)

print("\nVal accuracy:", val_acc)
print("Val macro F1:", val_f1)
print("Val macro ROC-AUC:", val_auc)

# ----- Classification report on validation -----
print("\nValidation classification report:")
print(classification_report(y_val, y_val_pred))

# ----- Confusion matrix on validation -----
cm = confusion_matrix(y_val, y_val_pred)
classes = np.unique(y_val)

plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion matrix (validation)")
plt.colorbar()
tick_marks = np.arange(len(classes))
plt.xticks(tick_marks, classes)
plt.yticks(tick_marks, classes)
plt.xlabel("Predicted label")
plt.ylabel("True label")

# Show values inside the matrix
thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            format(cm[i, j], "d"),
            horizontalalignment="center",
            color="white" if cm[i, j] > thresh else "black",
        )

plt.tight_layout()
plt.show()

# ----- ROC curves per class on validation -----
classes = np.unique(y_val)
y_val_bin = label_binarize(y_val, classes=classes)

plt.figure(figsize=(6, 5))
for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_val_bin[:, i], y_val_proba[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"Class {cls} (AUC = {roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves (validation)")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


### Error Analysis

In [ ]:
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from xgboost import XGBClassifier

# Learning-curve errors
train_errors = []
val_errors = []

# Training sizes
train_sizes = range(2000, len(X_train_opt), 2000)
print(list(train_sizes))

for n_train in train_sizes:
    # Subsample training data with stratification
    X_tr_subset, _, y_tr_subset, _ = train_test_split(
        X_train_opt,
        y_train,
        train_size=n_train,
        random_state=42,
        stratify=y_train
    )
    print(X_tr_subset.shape)
    print(X_val_opt.shape)

    # New XGBoost model with best hyperparameters
    lc_params = best_params.copy()
    lc_params.update({
        "objective": "multi:softprob",
        "num_class": len(np.unique(y_train)),
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "random_state": 42,
        "n_jobs": -1
    })
    model = XGBClassifier(**lc_params)

    # Train on subset
    model.fit(X_tr_subset, y_tr_subset, eval_set=[(X_val_opt, y_val)], verbose=False)

    # Predict probabilities
    y_tr_proba = model.predict_proba(X_tr_subset)
    y_val_proba = model.predict_proba(X_val_opt)

    # Log-loss on train/validation
    train_err = log_loss(y_tr_subset, y_tr_proba)
    val_err = log_loss(y_val, y_val_proba)

    train_errors.append(train_err)
    val_errors.append(val_err)

    print(f"Train size: {n_train}, Train log-loss: {train_err:.3f}, Val log-loss: {val_err:.3f}")

# Plot learning curve
plt.plot(train_sizes, train_errors, label="Train log-loss", marker="o")
plt.plot(train_sizes, val_errors, label="Validation log-loss", marker="o")
plt.xlabel("Training set size")
plt.ylabel("Log-loss")
plt.title("Learning Curve - XGBoost (Optuna-tuned)")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
import xgboost as xgb
import matplotlib.pyplot as plt
import numpy as np

#1. 定义 XGBoost 参数（
best_params = {
    'n_estimators': 2000,       # 大上限，配合 early stopping 使用
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.66,
    'colsample_bytree': 0.7,
    'min_child_weight': 20,
    'reg_alpha': 5.0,
    'reg_lambda': 10.0,
    'objective': 'multi:softprob',
    'random_state': 42,
    'n_jobs': -1
}

# 2. Error analysis：不同训练集大小的 log-loss 曲线
train_errors = []
val_errors = []

# 注意：这里用 X_train_final / y_train 做 error analysis
# 下界 2000、步长 2000 可以按你数据量调整
train_sizes = range(2000, len(X_train_final), 2000)
print("Train sizes:", list(train_sizes))

for n_train in train_sizes:
    X_train_sub, X_val_sub, y_train_sub, y_val_sub = train_test_split(
        X_train_final, y_train,
        train_size=n_train,
        random_state=42,
        stratify=y_train
    )

    print(f"\nCurrent train size: {n_train}")
    print("X_train_sub:", X_train_sub.shape)
    print("X_val_sub:", X_val_sub.shape)

    # 训练：每个 n_train 重新初始化一个模型
    model = xgb.XGBClassifier(**best_params)

    # 这里仍然用 early stopping，但只当作训练细节，
    # 逻辑上仍是“训练模型 + 计算 train/val log-loss”
    model.fit(
        X_train_sub, y_train_sub,
        eval_set=[(X_train_sub, y_train_sub), (X_val_sub, y_val_sub)],
        eval_metric='mlogloss',
        early_stopping_rounds=50,
        verbose=False
    )

    # 预测 train / val 概率
    y_train_pred_prob = model.predict_proba(X_train_sub)
    y_val_pred_prob = model.predict_proba(X_val_sub)

    # 计算 log-loss
    train_err = log_loss(y_train_sub, y_train_pred_prob)
    val_err = log_loss(y_val_sub, y_val_pred_prob)

    train_errors.append(train_err)
    val_errors.append(val_err)

    print(f"Train size: {n_train}, Train log-loss: {train_err:.3f}, Val log-loss: {val_err:.3f}")

# 3. 画学习曲线
plt.plot(list(train_sizes), train_errors, label='Train log-loss', marker='o')
plt.plot(list(train_sizes), val_errors, label='Validation log-loss', marker='o')
plt.xlabel("Training set size")
plt.ylabel("Log-loss")
plt.title("Learning Curve - XGBoost")
plt.legend()
plt.grid(True)
plt.show()


## Nueral Network

###NN+ V0-Feature

#### Model Building

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import make_scorer, f1_score, recall_score, precision_score
from sklearn.neural_network import MLPClassifier


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


# Pipeline with NN
pipeline = Pipeline([
    ('nn', MLPClassifier(
        hidden_layer_sizes=( 64, 32),  # Three hidden layers: 128 → 64 → 32 neurons
        activation='relu',                 # ReLU activation
        solver='adam',                     # Adam optimizer
        alpha=0.0001,                      # L2 regularization
        learning_rate_init=0.001,          # Initial learning rate
        max_iter=500,                      # Maximum iterations
        random_state=42,
        early_stopping=True,               # Early stopping to prevent overfitting
        n_iter_no_change=20,               # Stop if no improvement for 20 iterations
        verbose=True
    ))
])


#### Model Training & Model Evaluation

In [ ]:

scoring = {
    'accuracy': 'accuracy',
    'f1_micro': 'f1_micro',
    'f1_macro': 'f1_macro',
    'precision': 'precision_macro',
    'recall': 'recall_macro'
}

cv_results = cross_validate(
    pipeline, X_train_final, y_train_origin,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

# Print the average score for each evaluation metric.
for metric in scoring.keys():
    print(f"{metric}: {cv_results['test_' + metric].mean():.4f}")

In [ ]:
cv_results

#### Error Analysis

In [ ]:
from sklearn.metrics import log_loss, accuracy_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# errors
train_errors = []
val_errors = []

# train_sizes
train_sizes = range(2000, len(X_train),2000)  # 从 1 到 n-1
print(train_sizes)

for n_train in train_sizes:

    X_train1, X_val, y_train1, y_val = train_test_split(X_train, y_train, train_size=n_train, random_state=42, stratify=y_train)
    print(X_train1.shape)
    print(X_val.shape)

    # training
    # model = LogisticRegression(max_iter=50)
    # model.fit(X_train, y_train)
    pipeline.fit(X_train1, y_train1)

    # predict
    y_train_pred_prob =  pipeline.predict_proba(X_train1)
    y_val_pred_prob = pipeline.predict_proba(X_val)

    # log-loss
    train_err = log_loss(y_train1, y_train_pred_prob)
    val_err = log_loss(y_val, y_val_pred_prob)

    train_errors.append(train_err)
    val_errors.append(val_err)

    print(f"Train size: {n_train}, Train log-loss: {train_err:.3f}, Val log-loss: {val_err:.3f}")

# Plotting Learning Curve
plt.plot(train_sizes, train_errors, label='Train log-loss', marker='o')
plt.plot(train_sizes, val_errors, label='Validation log-loss', marker='o')
plt.xlabel("Training set size")
plt.ylabel("Log-loss")
plt.title("Learning Curve - Random Forest")
plt.legend()
plt.show()

### Feature Selection + Feature Ablation


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from joblib import Parallel, delayed

# -------------------------
# 参数设置
# -------------------------
RANDOM_SEED = 42
N_SPLITS = 5
AUC_THRESHOLD = 0.001
N_JOBS = -1  # 并行折数

# -------------------------
# 数据准备
# -------------------------
X = X_train_final  # if user has provided these names
y = y_train_origin
print("Using X_train_final / y_train_origin from environment.")

feature_names = X.columns.tolist()
n_classes = len(np.unique(y))
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}, Classes: {n_classes}")

# Binarize for multiclass ROC AUC (OvR macro)
y_bin = label_binarize(y, classes=np.arange(n_classes))

# Convert to numpy for speed (we will keep pandas for XGBoost fit to preserve feature names)
X_values = X.values
y_values = y





# -------------------------
# 1) 定义 CV 宏观 ROC AUC 函数
# -------------------------
def macro_roc_auc(y_true_bin, y_pred_prob):
    return roc_auc_score(y_true_bin, y_pred_prob, average='macro', multi_class='ovr')

# -------------------------
# 2) 训练初始 MLP 模型
# -------------------------
mlp_full = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500, random_state=RANDOM_SEED)
mlp_full.fit(X_values, y_values)

# -------------------------
# 3) 特征重要性 via Permutation Importance
# -------------------------
def permutation_importance(model, X_val, y_val_bin, metric_func, n_repeats=5):
    baseline = metric_func(y_val_bin, model.predict_proba(X_val))
    importances = np.zeros(X_val.shape[1])
    rng = np.random.default_rng(RANDOM_SEED)
    for i in range(X_val.shape[1]):
        scores = []
        for _ in range(n_repeats):
            X_perm = X_val.copy()
            rng.shuffle(X_perm[:, i])
            score = metric_func(y_val_bin, model.predict_proba(X_perm))
            scores.append(baseline - score)
        importances[i] = np.mean(scores)
    return importances

feature_importances = permutation_importance(mlp_full, X_values, y_bin, macro_roc_auc, n_repeats=5)
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances
}).sort_values("importance", ascending=False)

# 初步筛选 top-K 特征
TOP_K = min(30, X_values.shape[1])
selected_features = importance_df["feature"].iloc[:TOP_K].tolist()
print("Initial selected features:", selected_features)

# -------------------------
# 4) Feature Ablation
# -------------------------
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
splits = list(skf.split(X_values, y_values))
feat_to_idx = {f:i for i,f in enumerate(feature_names)}
remaining_idxs = [feat_to_idx[f] for f in selected_features]

auc_records = []

def train_mlp_and_auc(X_tr, y_tr, X_val, y_val_bin):
    model = MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=RANDOM_SEED)
    model.fit(X_tr, y_tr)
    y_pred = model.predict_proba(X_val)
    return macro_roc_auc(y_val_bin, y_pred)

for it in range(len(remaining_idxs)):
    cur_features = [feature_names[idx] for idx in remaining_idxs]
    # 移除最不重要特征
    if len(cur_features) > 1:
        cur_order = sorted(cur_features, key=lambda f: importance_df[importance_df["feature"]==f]["importance"].values[0], reverse=True)
        removed_feature = cur_order[-1]
        removed_idx = feat_to_idx[removed_feature]
        remaining_idxs = [i for i in remaining_idxs if i != removed_idx]
    else:
        removed_feature = None

    cur_idxs = np.array(remaining_idxs, dtype=int)

    # 并行 CV
    fold_aucs = Parallel(n_jobs=N_JOBS)(
        delayed(train_mlp_and_auc)(
            X_values[tr][:, cur_idxs],
            y_values[tr],
            X_values[va][:, cur_idxs],
            y_bin[va]
        ) for tr, va in splits
    )

    mean_auc = float(np.mean(fold_aucs))
    auc_records.append({
        "num_features": len(cur_idxs),
        "removed_feature": removed_feature,
        "mean_cv_auc": mean_auc,
        "remaining_features": [feature_names[idx] for idx in cur_idxs]
    })
    print(f"Iteration {it+1}: removed={removed_feature}, #features={len(cur_idxs)}, mean AUC={mean_auc:.4f}")

ablation_df = pd.DataFrame(auc_records)

# -------------------------
# 5) 选择最优特征子集
# -------------------------
max_auc = ablation_df["mean_cv_auc"].max()
optimal_row = ablation_df[ablation_df["mean_cv_auc"] >= (max_auc - AUC_THRESHOLD)].iloc[0]
optimal_features = optimal_row["remaining_features"]

print(f"Max mean CV AUC: {max_auc:.4f}")
print(f"Optimal feature set ({len(optimal_features)} features): {optimal_features}")


### Super parameters adjusting

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import numpy as np

# 1) Optimal feature set for Neural Network
NN_OPTIMAL_FEATURES = [
    'Outstanding_Debt', 'Delay_from_due_date', 'Interest_Rate',
    'Credit_Mix_Standard', 'Num_Bank_Accounts', 'Changed_Credit_Limit',
    'Num_Credit_Card', 'Annual_Income', 'Credit_Mix_Good',
    'Payment_of_Min_Amount_Yes', 'Num_of_Loan', 'Credit_History_Age',
    'Num_of_Delayed_Payment', 'Student Loan_True', 'Age',
    'Personal Loan_True', 'Credit-Builder Loan_True', 'Credit_Mix_Bad',
    'Debt Consolidation_True', 'Monthly_Inhand_Salary', 'Mortgage_True',
    'Payday Loan_True', 'Month_January', 'Month_March',
    'Home Equity Loan_True', 'Credit-Builder Loan_False',
    'Month_February', 'Debt Consolidation_False', 'Occupation_Engineer'
]

# 2) Train/validation data with optimal features
X_nn_train = X_train[NN_OPTIMAL_FEATURES].copy()
y_nn_train = y_train

print("NN train shape:", X_nn_train.shape)

# 3) Base MLP model and hyperparameter space
base_mlp = MLPClassifier(max_iter=200, random_state=RANDOM_SEED)

param_dist_mlp = {
    "hidden_layer_sizes": [
        (64,), (128,), (64, 32),
        (128, 64), (64, 64, 32)
    ],
    "activation": ["relu", "tanh"],
    "alpha": np.logspace(-5, -2, 4),
    "learning_rate_init": np.logspace(-4, -2, 3),
    "batch_size": [64, 128, 256]
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)

search_mlp = RandomizedSearchCV(
    estimator=base_mlp,
    param_distributions=param_dist_mlp,
    n_iter=30,
    scoring="roc_auc_ovr",
    cv=skf,
    n_jobs=N_JOBS,
    verbose=2,
    random_state=RANDOM_SEED
)

# 4) Hyperparameter tuning
print(f"\n[MLP] Starting hyperparameter tuning on {X_nn_train.shape} ...")
search_mlp.fit(X_nn_train, y_nn_train)

print("\n================ MLP Tuning Result ================")
print(f"Best CV macro ROC-AUC: {search_mlp.best_score_:.4f}")
print("Best params:", search_mlp.best_params_)

best_mlp = search_mlp.best_estimator_

# 5) Evaluation on validation set
X_val_nn = X_val[NN_OPTIMAL_FEATURES].copy()

y_val_pred = best_mlp.predict(X_val_nn)
y_val_proba = best_mlp.predict_proba(X_val_nn)

print("\nValidation classification report:")
print(classification_report(y_val, y_val_pred))

val_auc = roc_auc_score(y_val, y_val_proba, multi_class="ovr", average="macro")
print(f"Validation Macro ROC-AUC: {val_auc:.4f}")

cm = confusion_matrix(y_val, y_val_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(xticks_rotation=45)
plt.title("Optimized MLP Confusion Matrix")
plt.show()

# 6) Retrain final MLP on full training data
X_full_nn = X_train_final[NN_OPTIMAL_FEATURES].copy()
y_full_nn = y_train_origin

final_mlp = MLPClassifier(
    **search_mlp.best_params_,
    max_iter=300,
    random_state=RANDOM_SEED
)
final_mlp.fit(X_full_nn, y_full_nn)

print("[MLP] Final model retrained successfully on full data.")


### Error Analysis

In [ ]:
from sklearn.metrics import log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
import matplotlib.pyplot as plt
import numpy as np

# Feature matrices for NN (train / validation, optimal features only)
X_train_nn = X_train[NN_OPTIMAL_FEATURES].copy()
X_val_nn   = X_val[NN_OPTIMAL_FEATURES].copy()

print("X_train_nn shape:", X_train_nn.shape)
print("X_val_nn shape:", X_val_nn.shape)

# Learning curve containers
train_errors = []
val_errors = []

# Training set sizes
train_sizes = range(2000, len(X_train_nn), 2000)
print(list(train_sizes))

# Base params from tuned MLP
mlp_base_params = best_mlp.get_params()
mlp_base_params["random_state"] = RANDOM_SEED  # ensure reproducibility

for n_train in train_sizes:
    # Subsample training data with stratification
    X_tr_subset, _, y_tr_subset, _ = train_test_split(
        X_train_nn,
        y_train,
        train_size=n_train,
        random_state=RANDOM_SEED,
        stratify=y_train
    )
    print(X_tr_subset.shape)

    # New MLP with tuned hyperparameters
    mlp = MLPClassifier(**mlp_base_params)

    # Train on subset
    mlp.fit(X_tr_subset, y_tr_subset)

    # Predict probabilities
    y_tr_proba = mlp.predict_proba(X_tr_subset)
    y_val_proba = mlp.predict_proba(X_val_nn)

    # Log-loss on train and validation
    train_err = log_loss(y_tr_subset, y_tr_proba)
    val_err = log_loss(y_val, y_val_proba)

    train_errors.append(train_err)
    val_errors.append(val_err)

    print(f"Train size: {n_train}, Train log-loss: {train_err:.3f}, "
          f"Val log-loss: {val_err:.3f}")

# Plot learning curve
plt.plot(train_sizes, train_errors, label="Train log-loss", marker="o")
plt.plot(train_sizes, val_errors, label="Validation log-loss", marker="o")
plt.xlabel("Training set size")
plt.ylabel("Log-loss")
plt.title("Learning Curve - Tuned Neural Network (MLP)")
plt.legend()
plt.show()
